In [ ]:
# ==============================================================
# LOCAL PROJECT SETUP (added during VS Code migration)
# Portable replacement for Google Colab / Google Drive mounting.
# All research code below this block is unchanged.
# ==============================================================
import sys as _sys
from pathlib import Path as _Path

def _locate_project_root():
    start = _Path.cwd().resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "src" / "ecg_qpain" / "paths.py").is_file():
            return candidate
    return start

PROJECT_ROOT = _locate_project_root()
_src = str(PROJECT_ROOT / "src")
if _src not in _sys.path:
    _sys.path.insert(0, _src)

from ecg_qpain.paths import (
    ROOT, BASE_DIR, PROJECT_DIR, DRIVE_DIR, MYDRIVE, MOUNT_POINT, mount_path,
    DATA_DIR, RAW_DIR, PROCESSED_DIR, MODELS_DIR, REPORTS_DIR,
    MITDB_DIR, NSTDB_DIR, MITDB_EXTRACTED, NSTDB_EXTRACTED,
    B2_DIR, B3_DIR, B4_DIR,
)
print("Local project root:", PROJECT_ROOT)
# ==============================================================

# (Colab Drive import removed - local execution)
# (Colab Drive mount removed - local execution)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 2027

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = str(PROJECT_ROOT)

PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
MODEL_DIR = os.path.join(BASE_DIR, "models")

B2_DIR = os.path.join(
    MODEL_DIR,
    "B2_Noise_Augmented_ResNet"
)

B3_DIR = os.path.join(
    MODEL_DIR,
    "B3_CAM_Consistency"
)

B4_DIR = os.path.join(
    MODEL_DIR,
    "B4_Gated_CAM_Consistency"
)

os.makedirs(B3_DIR, exist_ok=True)
os.makedirs(B4_DIR, exist_ok=True)

print("Project paths initialized!")
print("Processed data:", os.path.exists(PROCESSED_DIR))
print("Reports:", os.path.exists(REPORT_DIR))
print("B2 model directory:", os.path.exists(B2_DIR))
print("B3 directory:", os.path.exists(B3_DIR))
print("B4 directory:", os.path.exists(B4_DIR))

In [ ]:
B2_MODEL_PATH = os.path.join(
    B2_DIR,
    "best_B2_resnet.keras"
)

assert os.path.exists(B2_MODEL_PATH)

b2_model = tf.keras.models.load_model(B2_MODEL_PATH)

print("B2 model loaded successfully!")
print("Input shape:", b2_model.input_shape)
print("Output shape:", b2_model.output_shape)

In [ ]:
b2_model.summary()

In [ ]:
print("CONVOLUTION LAYERS:\n")

for layer in b2_model.layers:
    if isinstance(
        layer,
        (tf.keras.layers.Conv1D, tf.keras.layers.SeparableConv1D)
    ):
        print(
            "Layer:",
            layer.name,
            "| Output shape:",
            layer.output.shape
        )

In [ ]:
conv_layers = [
    layer for layer in b2_model.layers
    if isinstance(
        layer,
        (tf.keras.layers.Conv1D, tf.keras.layers.SeparableConv1D)
    )
]

assert len(conv_layers) > 0, "No convolution layer found!"

LAST_CONV_LAYER = conv_layers[-1].name

print("Total convolution layers:", len(conv_layers))
print("Final convolution layer:", LAST_CONV_LAYER)

In [ ]:
CAM_LAYER_NAME = "activation_6"

grad_model = tf.keras.Model(
    inputs=b2_model.inputs,
    outputs=[
        b2_model.get_layer(CAM_LAYER_NAME).output,
        b2_model.output
    ]
)

print("Grad-CAM model created successfully!")
print("Target layer:", CAM_LAYER_NAME)
print("Feature output shape:",
      b2_model.get_layer(CAM_LAYER_NAME).output.shape)

In [ ]:
def generate_gradcam(model, input_signal, class_index=None):

    input_signal = tf.convert_to_tensor(
        input_signal, dtype=tf.float32
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = model(
            input_signal, training=False
        )

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        class_score = predictions[:, class_index]

    gradients = tape.gradient(class_score, conv_outputs)

    weights = tf.reduce_mean(
        gradients, axis=1, keepdims=True
    )

    cam = tf.reduce_sum(
        conv_outputs * weights, axis=-1
    )

    cam = tf.nn.relu(cam)

    cam = cam / (
        tf.reduce_max(cam, axis=1, keepdims=True) + 1e-8
    )

    cam = tf.expand_dims(cam, axis=-1)

    cam = tf.image.resize(cam, size=(288, 1))

    cam = tf.squeeze(cam, axis=-1)

    return cam.numpy(), predictions.numpy()

In [ ]:
sample_index = 0

sample_ecg = X_val[sample_index:sample_index+1]

cam_map, prediction = generate_gradcam(
    grad_model,
    sample_ecg
)

print("ECG shape:", sample_ecg.shape)
print("CAM shape:", cam_map.shape)
print("Prediction:", prediction)
print("Predicted class:", np.argmax(prediction))
print("CAM minimum:", cam_map.min())
print("CAM maximum:", cam_map.max())

assert cam_map.shape == (1, 288)
assert np.all(np.isfinite(cam_map))

print("GRAD-CAM TEST PASSED!")

In [ ]:
# Load processed ECG datasets

import os
import numpy as np

BASE_DIR = str(PROJECT_ROOT)
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")

# Load training data
train_data = np.load(
    os.path.join(PROCESSED_DIR, "train_beats.npz")
)

X_train = train_data["X"].astype(np.float32)
y_train = train_data["y"].astype(np.int32)

# Load validation data
val_data = np.load(
    os.path.join(PROCESSED_DIR, "validation_beats.npz")
)

X_val = val_data["X"].astype(np.float32)
y_val = val_data["y"].astype(np.int32)

# Add channel dimension
if X_train.ndim == 2:
    X_train = X_train[..., np.newaxis]

if X_val.ndim == 2:
    X_val = X_val[..., np.newaxis]

print("Training ECG:", X_train.shape)
print("Training labels:", y_train.shape)

print("Validation ECG:", X_val.shape)
print("Validation labels:", y_val.shape)

assert X_train.shape[1:] == (288, 1)
assert X_val.shape[1:] == (288, 1)

print("\nDATA LOADING PASSED!")

In [ ]:
# Corrected Grad-CAM Function

def generate_gradcam(model, input_signal, class_index=None):

    input_signal = tf.convert_to_tensor(
        input_signal,
        dtype=tf.float32
    )

    if len(input_signal.shape) == 2:
        input_signal = tf.expand_dims(input_signal, axis=0)

    with tf.GradientTape() as tape:

        conv_outputs, predictions = model(
            input_signal,
            training=False
        )

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        class_score = predictions[:, class_index]

    gradients = tape.gradient(
        class_score,
        conv_outputs
    )

    if gradients is None:
        raise ValueError("Grad-CAM gradients are None!")

    weights = tf.reduce_mean(
        gradients,
        axis=1,
        keepdims=True
    )

    cam = tf.reduce_sum(
        conv_outputs * weights,
        axis=-1
    )

    cam = tf.nn.relu(cam)

    # Normalize CAM
    cam_max = tf.reduce_max(
        cam,
        axis=1,
        keepdims=True
    )

    cam = cam / (cam_max + 1e-8)

    # Correct 1D CAM resizing
    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1),
        method="bilinear"
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam.numpy(), predictions.numpy()

In [ ]:
sample_index = 0

sample_ecg = X_val[sample_index:sample_index+1]

cam_map, prediction = generate_gradcam(
    grad_model,
    sample_ecg
)

print("ECG shape:", sample_ecg.shape)
print("CAM shape:", cam_map.shape)

print("Prediction:", prediction)
print("Predicted class:", np.argmax(prediction))

print("CAM minimum:", cam_map.min())
print("CAM maximum:", cam_map.max())

print("Finite values:", np.all(np.isfinite(cam_map)))

assert cam_map.shape == (1, 288)
assert np.all(np.isfinite(cam_map))

print("\nCAM SHAPE VALIDATION PASSED!")

In [ ]:
if cam_map.max() <= 1e-8:
    print("WARNING: CAM is all zero. Attribution needs investigation.")
else:
    print("CAM has non-zero attribution values.")
    print("Non-zero CAM points:", np.count_nonzero(cam_map))
    print("CAM standard deviation:", np.std(cam_map))


In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(2, 1, 1)
plt.plot(sample_ecg[0, :, 0])
plt.title("Original ECG Signal")
plt.ylabel("Amplitude")

plt.subplot(2, 1, 2)
plt.plot(cam_map[0])
plt.title("Grad-CAM Attribution Map")
plt.xlabel("Sample Index")
plt.ylabel("Attribution")

plt.tight_layout()
plt.show()

In [ ]:
config_path = os.path.join(
    REPORT_DIR,
    "noise_generation_config.json"
)

with open(config_path, "r") as f:
    noise_config = json.load(f)

print(json.dumps(noise_config, indent=2)[:3000])

In [ ]:
import wfdb

NSTDB_DIR = os.path.join(
    BASE_DIR,
    "data/raw/nstdb/mit-bih-noise-stress-test-database-1.0.0"
)

noise_signals = {}

for noise_type in ["bw", "ma"]:

    signal, fields = wfdb.rdsamp(
        os.path.join(NSTDB_DIR, noise_type)
    )

    noise_signals[noise_type] = signal[:, 0].astype(np.float32)

    print(
        noise_type.upper(),
        "Shape:", signal.shape,
        "Sampling frequency:", fields["fs"]
    )

In [ ]:
# (pip install moved to requirements.txt; run: pip install -r requirements.txt)


In [ ]:
import wfdb

print("WFDB version:", wfdb.__version__)
print("WFDB imported successfully!")

In [ ]:
NSTDB_DIR = os.path.join(
    BASE_DIR,
    "data/raw/nstdb/mit-bih-noise-stress-test-database-1.0.0"
)

noise_signals = {}

for noise_type in ["bw", "ma"]:

    signal, fields = wfdb.rdsamp(
        os.path.join(NSTDB_DIR, noise_type)
    )

    noise_signals[noise_type] = signal[:, 0].astype(np.float32)

    print(
        noise_type.upper(),
        "Shape:", signal.shape,
        "Sampling frequency:", fields["fs"]
    )

In [ ]:
noise_bank = tf.constant(
    np.stack([
        noise_signals["bw"],
        noise_signals["ma"]
    ]),
    dtype=tf.float32
)

print("Noise bank shape:", noise_bank.shape)

assert noise_bank.shape == (2, 650000)

print("NOISE BANK VALIDATION PASSED!")

In [ ]:
FS = 360
BEAT_LENGTH = 288

TRAIN_SNRS = [24, 18, 12, 6]

TRAIN_START = 0
TRAIN_END = 390000


def add_random_noise(ecg):

    ecg = tf.cast(ecg, tf.float32)

    ecg = tf.squeeze(ecg, axis=-1)

    # Randomly select BW or MA noise
    noise_index = tf.random.uniform(
        [],
        0,
        2,
        dtype=tf.int32
    )

    # Select a random noise segment
    start = tf.random.uniform(
        [],
        TRAIN_START,
        TRAIN_END - BEAT_LENGTH + 1,
        dtype=tf.int32
    )

    indices = tf.range(start, start + BEAT_LENGTH)

    noise = tf.gather(
        noise_bank[noise_index],
        indices
    )

    noise = noise - tf.reduce_mean(noise)

    # Randomly select SNR
    snr_index = tf.random.uniform(
        [],
        0,
        len(TRAIN_SNRS),
        dtype=tf.int32
    )

    snr_db = tf.gather(
        tf.constant(TRAIN_SNRS, dtype=tf.float32),
        snr_index
    )

    # Calculate signal and noise power
    signal_power = tf.reduce_mean(tf.square(ecg))

    noise_power = tf.reduce_mean(tf.square(noise))

    target_noise_power = (
        signal_power / tf.pow(10.0, snr_db / 10.0)
    )

    scale = tf.sqrt(
        target_noise_power / tf.maximum(noise_power, 1e-12)
    )

    # Add noise
    noisy_ecg = ecg + noise * scale

    return tf.expand_dims(noisy_ecg, axis=-1)


print("Noise augmentation function created!")

In [ ]:
test_clean = tf.constant(X_train[:1])

test_noisy = add_random_noise(test_clean)

print("Clean shape:", test_clean.shape)
print("Noisy shape:", test_noisy.shape)

print(
    "Noisy finite:",
    bool(tf.reduce_all(
        tf.math.is_finite(test_noisy)
    ).numpy())
)

assert test_noisy.shape == (1, 288, 1)

assert bool(
    tf.reduce_all(
        tf.math.is_finite(test_noisy)
    ).numpy()
)

print("NOISE AUGMENTATION TEST PASSED!")


In [ ]:
CAM_LAYER_NAME = "activation_6"

grad_model = tf.keras.Model(
    inputs=b2_model.inputs,
    outputs=[
        b2_model.get_layer(CAM_LAYER_NAME).output,
        b2_model.output
    ]
)

print("Differentiable Grad-CAM model created!")
print("CAM feature shape:", b2_model.get_layer(CAM_LAYER_NAME).output.shape)

In [ ]:
def differentiable_cam(x, labels):

    with tf.GradientTape() as inner_tape:

        features, predictions = grad_model(
            x,
            training=True
        )

        inner_tape.watch(features)

        class_scores = tf.gather(
            predictions,
            labels,
            axis=1,
            batch_dims=1
        )

    gradients = inner_tape.gradient(
        tf.reduce_sum(class_scores),
        features
    )

    if gradients is None:
        raise ValueError("CAM gradients are None!")

    weights = tf.reduce_mean(
        gradients,
        axis=1,
        keepdims=True
    )

    cam = tf.reduce_sum(
        features * weights,
        axis=-1
    )

    cam = tf.nn.relu(cam)

    cam = cam / (
        tf.reduce_max(cam, axis=1, keepdims=True) + 1e-8
    )

    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1),
        method="bilinear"
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam, predictions

In [ ]:
test_cam, test_predictions = differentiable_cam(
    tf.constant(X_train[:2]),
    tf.constant(y_train[:2])
)

print("CAM shape:", test_cam.shape)
print("Prediction shape:", test_predictions.shape)
print("CAM finite:", bool(tf.reduce_all(tf.math.is_finite(test_cam)).numpy()))

assert test_cam.shape == (2, 288)
assert test_predictions.shape == (2, 4)

print("DIFFERENTIABLE CAM TEST PASSED!")

In [ ]:
CAM_LAYER_NAME = "activation_6"

cam_layer = b2_model.get_layer(CAM_LAYER_NAME)

feature_extractor = tf.keras.Model(
    inputs=b2_model.inputs,
    outputs=cam_layer.output
)

classifier_head = tf.keras.Model(
    inputs=cam_layer.output,
    outputs=b2_model.output
)

print("Feature extractor:", feature_extractor.output_shape)
print("Classifier head:", classifier_head.output_shape)

print("CAM model separation completed!")

In [ ]:
def differentiable_cam(x, labels):

    with tf.GradientTape() as inner_tape:

        # Extract intermediate features
        features = feature_extractor(
            x,
            training=True
        )

        # Watch features BEFORE classifier calculation
        inner_tape.watch(features)

        # Calculate predictions from features
        predictions = classifier_head(
            features,
            training=True
        )

        # Select true class scores
        class_scores = tf.gather(
            predictions,
            labels,
            axis=1,
            batch_dims=1
        )

        total_score = tf.reduce_sum(class_scores)

    gradients = inner_tape.gradient(
        total_score,
        features
    )

    if gradients is None:
        raise ValueError("CAM gradients are still None!")

    weights = tf.reduce_mean(
        gradients,
        axis=1,
        keepdims=True
    )

    cam = tf.reduce_sum(
        features * weights,
        axis=-1
    )

    cam = tf.nn.relu(cam)

    cam = cam / (
        tf.reduce_max(cam, axis=1, keepdims=True) + 1e-8
    )

    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1),
        method="bilinear"
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam, predictions

In [ ]:
test_cam, test_predictions = differentiable_cam(
    tf.constant(X_train[:2]),
    tf.constant(y_train[:2])
)

print("CAM shape:", test_cam.shape)
print("Prediction shape:", test_predictions.shape)

print(
    "CAM finite:",
    bool(tf.reduce_all(tf.math.is_finite(test_cam)).numpy())
)

assert test_cam.shape == (2, 288)
assert test_predictions.shape == (2, 4)

print("DIFFERENTIABLE CAM TEST PASSED!")

In [ ]:
with tf.GradientTape() as outer_tape:

    clean_cam, clean_predictions = differentiable_cam(
        clean_batch,
        label_batch
    )

    noisy_cam, noisy_predictions = differentiable_cam(
        noisy_batch,
        label_batch
    )

    test_cam_loss = cam_consistency_loss(
        clean_cam,
        noisy_cam
    )

gradients = outer_tape.gradient(
    test_cam_loss,
    b2_model.trainable_variables
)

connected_gradients = [
    g for g in gradients if g is not None
]

print("CAM consistency loss:", float(test_cam_loss.numpy()))

print(
    "Connected gradients:",
    len(connected_gradients),
    "/",
    len(b2_model.trainable_variables)
)

print(
    "Gradient finite:",
    all(
        bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
        for g in connected_gradients
    )
)

assert len(connected_gradients) > 0

assert all(
    bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
    for g in connected_gradients
)

print("SECOND-ORDER GRADIENT TEST PASSED!")

In [ ]:
BATCH_SIZE = 4

clean_batch = tf.constant(
    X_train[:BATCH_SIZE],
    dtype=tf.float32
)

label_batch = tf.constant(
    y_train[:BATCH_SIZE],
    dtype=tf.int32
)

noisy_batch = tf.map_fn(
    lambda x: tf.squeeze(
        add_random_noise(tf.expand_dims(x, axis=0)),
        axis=0
    ),
    clean_batch,
    fn_output_signature=tf.float32
)

print("Clean batch:", clean_batch.shape)
print("Noisy batch:", noisy_batch.shape)
print("Label batch:", label_batch.shape)

assert clean_batch.shape == (4, 288, 1)
assert noisy_batch.shape == (4, 288, 1)

print("BATCH RECREATION PASSED!")

In [ ]:
with tf.GradientTape() as outer_tape:

    clean_cam, clean_predictions = differentiable_cam(
        clean_batch,
        label_batch
    )

    noisy_cam, noisy_predictions = differentiable_cam(
        noisy_batch,
        label_batch
    )

    test_cam_loss = cam_consistency_loss(
        clean_cam,
        noisy_cam
    )

gradients = outer_tape.gradient(
    test_cam_loss,
    b2_model.trainable_variables
)

connected_gradients = [
    g for g in gradients if g is not None
]

print("CAM consistency loss:", float(test_cam_loss.numpy()))

print(
    "Connected gradients:",
    len(connected_gradients),
    "/",
    len(b2_model.trainable_variables)
)

print(
    "Gradient finite:",
    all(
        bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
        for g in connected_gradients
    )
)

assert len(connected_gradients) > 0

assert all(
    bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
    for g in connected_gradients
)

print("SECOND-ORDER GRADIENT TEST PASSED!")

In [ ]:
def cam_consistency_loss(clean_cam, noisy_cam):

    return tf.reduce_mean(
        tf.square(clean_cam - noisy_cam)
    )

print("CAM consistency loss function created!")

In [ ]:
with tf.GradientTape() as outer_tape:

    clean_cam, clean_predictions = differentiable_cam(
        clean_batch,
        label_batch
    )

    noisy_cam, noisy_predictions = differentiable_cam(
        noisy_batch,
        label_batch
    )

    test_cam_loss = cam_consistency_loss(
        clean_cam,
        noisy_cam
    )

gradients = outer_tape.gradient(
    test_cam_loss,
    b2_model.trainable_variables
)

connected_gradients = [
    g for g in gradients if g is not None
]

print("CAM consistency loss:", float(test_cam_loss.numpy()))

print(
    "Connected gradients:",
    len(connected_gradients),
    "/",
    len(b2_model.trainable_variables)
)

print(
    "Gradient finite:",
    all(
        bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
        for g in connected_gradients
    )
)

assert len(connected_gradients) > 0

assert all(
    bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
    for g in connected_gradients
)

print("SECOND-ORDER GRADIENT TEST PASSED!")

In [ ]:
import tensorflow as tf
import os

B3_DIR = os.path.join(MODEL_DIR, "B3_CAM_Consistency")
B4_DIR = os.path.join(MODEL_DIR, "B4_Gated_CAM_Consistency")

os.makedirs(B3_DIR, exist_ok=True)
os.makedirs(B4_DIR, exist_ok=True)

# Create independent models from B2
b3_model = tf.keras.models.clone_model(b2_model)
b3_model.set_weights(b2_model.get_weights())

b4_model = tf.keras.models.clone_model(b2_model)
b4_model.set_weights(b2_model.get_weights())

print("B3 parameters:", b3_model.count_params())
print("B4 parameters:", b4_model.count_params())

print("B3 and B4 models created successfully!")

In [ ]:
LEARNING_RATE = 0.0001

LAMBDA_CAM = 0.1

EPOCHS_B3 = 10
EPOCHS_B4 = 10

BATCH_SIZE = 32

print("Learning rate:", LEARNING_RATE)
print("CAM lambda:", LAMBDA_CAM)
print("B3 epochs:", EPOCHS_B3)
print("B4 epochs:", EPOCHS_B4)

print("TRAINING CONFIGURATION READY!")

In [ ]:
def noise_severity_gate(snr_db):

    severity = (24.0 - snr_db) / 18.0

    return tf.clip_by_value(
        severity,
        0.0,
        1.0
    )


for snr in [24, 18, 12, 6]:

    print(
        f"SNR {snr} dB:",
        float(noise_severity_gate(
            tf.constant(snr, dtype=tf.float32)
        ).numpy())
    )

In [ ]:
import json

experiment_config = {
    "project": "ECG_QPAIN_2027",
    "B3": "ResNet + CAM Consistency",
    "B4": "ResNet + Noise Severity Gated CAM Consistency",
    "learning_rate": LEARNING_RATE,
    "lambda_cam": LAMBDA_CAM,
    "epochs_B3": EPOCHS_B3,
    "epochs_B4": EPOCHS_B4,
    "batch_size": BATCH_SIZE,
    "severity_gate": "(24-SNR)/18",
    "training_SNRs": [24, 18, 12, 6],
    "initial_weights": "B2 trained weights",
    "test_data_used_for_training": False
}

config_path = os.path.join(
    REPORT_DIR,
    "B3_B4_experiment_config.json"
)

with open(config_path, "w") as f:
    json.dump(experiment_config, f, indent=4)

print("Configuration saved:", config_path)
print("EXPERIMENT CONFIGURATION SAVED!")

In [ ]:
print("B3 exists:", "b3_model" in globals())
print("B4 exists:", "b4_model" in globals())

assert "b3_model" in globals()
assert "b4_model" in globals()

print("B3 parameters:", b3_model.count_params())
print("B4 parameters:", b4_model.count_params())

print("B3-B4 MODEL VALIDATION PASSED!")

In [ ]:
def create_cam_components(model):

    feature_extractor = tf.keras.Model(
        inputs=model.inputs,
        outputs=model.get_layer("activation_6").output
    )

    classifier_head = tf.keras.Model(
        inputs=model.get_layer("activation_6").output,
        outputs=model.output
    )

    return feature_extractor, classifier_head


b3_feature_extractor, b3_classifier_head = (
    create_cam_components(b3_model)
)

b4_feature_extractor, b4_classifier_head = (
    create_cam_components(b4_model)
)

print("B3 CAM components:", b3_feature_extractor.output_shape)
print("B4 CAM components:", b4_feature_extractor.output_shape)

print("SEPARATE CAM COMPONENTS CREATED!")

In [ ]:
def differentiable_cam_for_model(
    x,
    labels,
    feature_extractor,
    classifier_head
):

    with tf.GradientTape() as inner_tape:

        features = feature_extractor(
            x,
            training=True
        )

        inner_tape.watch(features)

        predictions = classifier_head(
            features,
            training=True
        )

        class_scores = tf.gather(
            predictions,
            labels,
            axis=1,
            batch_dims=1
        )

        total_score = tf.reduce_sum(class_scores)

    gradients = inner_tape.gradient(
        total_score,
        features
    )

    if gradients is None:
        raise ValueError("CAM gradients are None!")

    weights = tf.reduce_mean(
        gradients,
        axis=1,
        keepdims=True
    )

    cam = tf.reduce_sum(
        features * weights,
        axis=-1
    )

    cam = tf.nn.relu(cam)

    cam = cam / (
        tf.reduce_max(cam, axis=1, keepdims=True) + 1e-8
    )

    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1),
        method="bilinear"
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam, predictions


print("MODEL-SPECIFIC CAM FUNCTION CREATED!")

In [ ]:
b3_test_cam, b3_test_pred = differentiable_cam_for_model(
    clean_batch,
    label_batch,
    b3_feature_extractor,
    b3_classifier_head
)

b4_test_cam, b4_test_pred = differentiable_cam_for_model(
    clean_batch,
    label_batch,
    b4_feature_extractor,
    b4_classifier_head
)

print("B3 CAM:", b3_test_cam.shape)
print("B4 CAM:", b4_test_cam.shape)

print("B3 predictions:", b3_test_pred.shape)
print("B4 predictions:", b4_test_pred.shape)

assert b3_test_cam.shape == (4, 288)
assert b4_test_cam.shape == (4, 288)

print("B3-B4 CAM TEST PASSED!")

In [ ]:
def add_random_noise_with_snr(ecg):

    ecg = tf.cast(ecg, tf.float32)
    ecg = tf.squeeze(ecg, axis=-1)

    # Select BW or MA
    noise_index = tf.random.uniform(
        [], 0, 2, dtype=tf.int32
    )

    # Select noise segment from training region
    start = tf.random.uniform(
        [],
        TRAIN_START,
        TRAIN_END - BEAT_LENGTH + 1,
        dtype=tf.int32
    )

    indices = tf.range(start, start + BEAT_LENGTH)

    noise = tf.gather(
        noise_bank[noise_index],
        indices
    )

    noise = noise - tf.reduce_mean(noise)

    # Select SNR
    snr_index = tf.random.uniform(
        [], 0, len(TRAIN_SNRS), dtype=tf.int32
    )

    snr_db = tf.gather(
        tf.constant(TRAIN_SNRS, dtype=tf.float32),
        snr_index
    )

    signal_power = tf.reduce_mean(tf.square(ecg))
    noise_power = tf.reduce_mean(tf.square(noise))

    target_noise_power = (
        signal_power / tf.pow(10.0, snr_db / 10.0)
    )

    scale = tf.sqrt(
        target_noise_power /
        tf.maximum(noise_power, 1e-12)
    )

    noisy_ecg = ecg + noise * scale

    return (
        tf.expand_dims(noisy_ecg, axis=-1),
        snr_db
    )


print("SNR-aware noise augmentation created!")



In [ ]:
test_clean = tf.constant(X_train[:1])

test_noisy, test_snr = add_random_noise_with_snr(
    test_clean[0]
)

print("Clean shape:", test_clean.shape)
print("Noisy shape:", test_noisy.shape)
print("Selected SNR:", float(test_snr.numpy()), "dB")

assert test_noisy.shape == (288, 1)
assert float(test_snr.numpy()) in TRAIN_SNRS

print("SNR-AWARE AUGMENTATION PASSED!")

In [ ]:
test_snrs = tf.constant(
    [24.0, 18.0, 12.0, 6.0],
    dtype=tf.float32
)

test_gates = noise_severity_gate(test_snrs)

print("SNR values:", test_snrs.numpy())
print("Gate values:", test_gates.numpy())

assert np.allclose(
    test_gates.numpy(),
    [0.0, 1/3, 2/3, 1.0]
)

print("SEVERITY GATE VALIDATION PASSED!")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_tensor = tf.constant(
    class_weights_array,
    dtype=tf.float32
)

print("Class weights:", class_weights_array)

print("CLASS WEIGHTS CREATED!")

In [ ]:
LAMBDA_CAM = 0.1

def calculate_b3_loss(
    labels,
    predictions,
    clean_cam,
    noisy_cam
):

    ce_loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        predictions
    )

    sample_weights = tf.gather(
        class_weights_tensor,
        labels
    )

    weighted_ce = tf.reduce_mean(
        ce_loss * sample_weights
    )

    cam_loss = cam_consistency_loss(
        clean_cam,
        noisy_cam
    )

    total_loss = weighted_ce + LAMBDA_CAM * cam_loss

    return total_loss, weighted_ce, cam_loss

In [ ]:
b3_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0001
)

with tf.GradientTape() as outer_tape:

    clean_cam, clean_predictions = differentiable_cam_for_model(
        clean_batch,
        label_batch,
        b3_feature_extractor,
        b3_classifier_head
    )

    noisy_cam, _ = differentiable_cam_for_model(
        noisy_batch,
        label_batch,
        b3_feature_extractor,
        b3_classifier_head
    )

    total_loss, ce_loss, cam_loss = calculate_b3_loss(
        label_batch,
        clean_predictions,
        clean_cam,
        noisy_cam
    )

b3_gradients = outer_tape.gradient(
    total_loss,
    b3_model.trainable_variables
)

b3_connected = [
    g for g in b3_gradients if g is not None
]

print("Total loss:", float(total_loss.numpy()))
print("CE loss:", float(ce_loss.numpy()))
print("CAM loss:", float(cam_loss.numpy()))

print(
    "Connected gradients:",
    len(b3_connected),
    "/",
    len(b3_model.trainable_variables)
)

assert len(b3_connected) > 0

assert all(
    bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
    for g in b3_connected
)

print("B3 GRADIENT TEST PASSED!")

In [ ]:
b2_weights = b2_model.get_weights()
b3_weights = b3_model.get_weights()

same_weights = all(
    np.array_equal(a, b)
    for a, b in zip(b2_weights, b3_weights)
)

print("B2 and B3 initially identical:", same_weights)

assert same_weights

print("INITIAL WEIGHT VALIDATION PASSED!")

In [ ]:
# Restore B3 with original B2 weights
b3_model.set_weights(
    b2_model.get_weights()
)

print("B3 weights restored from B2!")

In [ ]:
b2_weights = b2_model.get_weights()
b3_weights = b3_model.get_weights()

same_weights = all(
    np.array_equal(a, b)
    for a, b in zip(b2_weights, b3_weights)
)

print("B2 and B3 initially identical:", same_weights)

assert same_weights

print("INITIAL WEIGHT VALIDATION PASSED!")

In [ ]:
b4_model.set_weights(
    b2_model.get_weights()
)

b4_weights = b4_model.get_weights()

b4_same = all(
    np.array_equal(a, b)
    for a, b in zip(b2_weights, b4_weights)
)

print("B2 and B4 initially identical:", b4_same)

assert b4_same

print("B4 INITIAL WEIGHT VALIDATION PASSED!")

In [ ]:
def differentiable_cam_for_model(
    x,
    labels,
    feature_extractor,
    classifier_head
):

    with tf.GradientTape() as inner_tape:

        features = feature_extractor(
            x,
            training=False
        )

        inner_tape.watch(features)

        predictions = classifier_head(
            features,
            training=False
        )

        class_scores = tf.gather(
            predictions,
            labels,
            axis=1,
            batch_dims=1
        )

        total_score = tf.reduce_sum(class_scores)

    gradients = inner_tape.gradient(
        total_score,
        features
    )

    if gradients is None:
        raise ValueError("CAM gradients are None!")

    weights = tf.reduce_mean(
        gradients,
        axis=1,
        keepdims=True
    )

    cam = tf.reduce_sum(
        features * weights,
        axis=-1
    )

    cam = tf.nn.relu(cam)

    cam = cam / (
        tf.reduce_max(cam, axis=1, keepdims=True) + 1e-8
    )

    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1),
        method="bilinear"
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam, predictions

In [ ]:
b3_before = [
    w.copy() for w in b3_model.get_weights()
]

b4_before = [
    w.copy() for w in b4_model.get_weights()
]

# Test B3 CAM
_ = differentiable_cam_for_model(
    clean_batch,
    label_batch,
    b3_feature_extractor,
    b3_classifier_head
)

# Test B4 CAM
_ = differentiable_cam_for_model(
    clean_batch,
    label_batch,
    b4_feature_extractor,
    b4_classifier_head
)

b3_after = b3_model.get_weights()
b4_after = b4_model.get_weights()

b3_unchanged = all(
    np.array_equal(a, b)
    for a, b in zip(b3_before, b3_after)
)

b4_unchanged = all(
    np.array_equal(a, b)
    for a, b in zip(b4_before, b4_after)
)

print("B3 weights unchanged:", b3_unchanged)
print("B4 weights unchanged:", b4_unchanged)

assert b3_unchanged
assert b4_unchanged

print("BATCH NORMALIZATION CHECK PASSED!")

In [ ]:
with tf.GradientTape() as outer_tape:

    clean_cam, clean_predictions = differentiable_cam_for_model(
        clean_batch,
        label_batch,
        b3_feature_extractor,
        b3_classifier_head
    )

    noisy_cam, _ = differentiable_cam_for_model(
        noisy_batch,
        label_batch,
        b3_feature_extractor,
        b3_classifier_head
    )

    total_loss, ce_loss, cam_loss = calculate_b3_loss(
        label_batch,
        clean_predictions,
        clean_cam,
        noisy_cam
    )

b3_gradients = outer_tape.gradient(
    total_loss,
    b3_model.trainable_variables
)

b3_connected = [
    g for g in b3_gradients if g is not None
]

print("Total loss:", float(total_loss.numpy()))
print("CE loss:", float(ce_loss.numpy()))
print("CAM loss:", float(cam_loss.numpy()))

print(
    "Connected gradients:",
    len(b3_connected),
    "/",
    len(b3_model.trainable_variables)
)

assert len(b3_connected) > 0

assert all(
    bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
    for g in b3_connected
)

print("B3 FINAL GRADIENT TEST PASSED!")

In [ ]:
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(10000, seed=2027)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training batches:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Training dataset created!")

In [ ]:
def train_b3_step(clean_x, labels):

    noisy_x, _ = tf.map_fn(
        lambda x: add_random_noise_with_snr(x),
        clean_x,
        fn_output_signature=(
            tf.TensorSpec(shape=(288, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )

    with tf.GradientTape() as outer_tape:

        clean_cam, clean_predictions = differentiable_cam_for_model(
            clean_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        noisy_cam, _ = differentiable_cam_for_model(
            noisy_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        total_loss, ce_loss, cam_loss = calculate_b3_loss(
            labels,
            clean_predictions,
            clean_cam,
            noisy_cam
        )

    gradients = outer_tape.gradient(
        total_loss,
        b3_model.trainable_variables
    )

    gradient_variable_pairs = [
        (g, v)
        for g, v in zip(gradients, b3_model.trainable_variables)
        if g is not None
    ]

    b3_optimizer.apply_gradients(gradient_variable_pairs)

    return total_loss, ce_loss, cam_loss

In [ ]:
sample_x, sample_y = next(iter(train_dataset))

loss, ce, cam = train_b3_step(
    sample_x,
    sample_y
)

print("Training batch shape:", sample_x.shape)
print("Total loss:", float(loss.numpy()))
print("CE loss:", float(ce.numpy()))
print("CAM loss:", float(cam.numpy()))

print("B3 ONE-BATCH TRAINING PASSED!")

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Validation batches:", tf.data.experimental.cardinality(val_dataset).numpy())
print("VALIDATION DATASET READY!")

In [ ]:
gpus = tf.config.list_physical_devices("GPU")

print("Available GPUs:", gpus)

if len(gpus) > 0:
    print("GPU AVAILABLE!")
else:
    print("NO GPU DETECTED!")

In [ ]:
# Restore B3 from pretrained B2 weights
b3_model.set_weights(b2_model.get_weights())

# Reset optimizer
b3_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0001
)

# Verify weights
same_weights = all(
    np.array_equal(a, b)
    for a, b in zip(b2_model.get_weights(), b3_model.get_weights())
)

print("B3 restored from B2:", same_weights)
print("B3 optimizer reset!")

In [ ]:
LAMBDA_CAM = 0.1

def weighted_ce_loss(labels, predictions):
    ce = tf.keras.losses.sparse_categorical_crossentropy(
        labels, predictions
    )
    sample_weights = tf.gather(class_weights_tensor, labels)
    return tf.reduce_mean(ce * sample_weights)


def calculate_b3_loss(
    labels,
    clean_predictions,
    noisy_predictions,
    clean_cam,
    noisy_cam
):
    clean_ce = weighted_ce_loss(labels, clean_predictions)
    noisy_ce = weighted_ce_loss(labels, noisy_predictions)

    ce_loss = (clean_ce + noisy_ce) / 2.0

    cam_loss = tf.reduce_mean(
        tf.square(clean_cam - noisy_cam)
    )

    total_loss = ce_loss + LAMBDA_CAM * cam_loss

    return total_loss, ce_loss, cam_loss


print("B3 corrected loss function ready!")

In [ ]:
def train_b3_step(clean_x, labels):

    noisy_x, snr_values = tf.map_fn(
        lambda x: add_random_noise_with_snr(x),
        clean_x,
        fn_output_signature=(
            tf.TensorSpec(shape=(288, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )

    with tf.GradientTape() as outer_tape:

        clean_cam, clean_predictions = differentiable_cam_for_model(
            clean_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        noisy_cam, noisy_predictions = differentiable_cam_for_model(
            noisy_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        total_loss, ce_loss, cam_loss = calculate_b3_loss(
            labels,
            clean_predictions,
            noisy_predictions,
            clean_cam,
            noisy_cam
        )

    gradients = outer_tape.gradient(
        total_loss,
        b3_model.trainable_variables
    )

    gradient_pairs = [
        (g, v)
        for g, v in zip(gradients, b3_model.trainable_variables)
        if g is not None
    ]

    b3_optimizer.apply_gradients(gradient_pairs)

    return total_loss, ce_loss, cam_loss


print("B3 corrected training function ready!")

In [ ]:
BATCH_SIZE = 8

train_dataset_b3 = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset_b3 = (
    train_dataset_b3
    .shuffle(10000, seed=2027)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

sample_x, sample_y = next(iter(train_dataset_b3))

loss, ce, cam = train_b3_step(sample_x, sample_y)

print("Training batch shape:", sample_x.shape)
print("Total loss:", float(loss.numpy()))
print("CE loss:", float(ce.numpy()))
print("CAM loss:", float(cam.numpy()))
print("B3 CORRECTED TRAINING TEST PASSED!")

In [ ]:
import time

BATCH_SIZE = 8
B3_EPOCHS = 10
B3_LEARNING_RATE = 0.0001

print("Training device:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU")
print("Batch size:", BATCH_SIZE)
print("Epochs:", B3_EPOCHS)
print("Learning rate:", B3_LEARNING_RATE)

In [ ]:
print("Available noise signals:", list(noise_signals.keys()))
print("BW shape:", noise_signals["bw"].shape)
print("MA shape:", noise_signals["ma"].shape)

In [ ]:
VAL_START = 390000
VAL_END = 520000

VAL_SNRS = [18, 12, 6]
BEAT_LENGTH = 288

print("Validation region:", VAL_START, "to", VAL_END)
print("Validation SNRs:", VAL_SNRS)
print("Validation noise length:", VAL_END - VAL_START)

In [ ]:
def add_validation_noise(clean_signal, noise_signal, snr_db):
    clean_signal = np.asarray(clean_signal, dtype=np.float32)
    noise_signal = np.asarray(noise_signal, dtype=np.float32)

    noise_signal = noise_signal - np.mean(noise_signal)

    signal_power = np.mean(clean_signal ** 2)
    noise_power = np.mean(noise_signal ** 2)

    if signal_power <= 0 or noise_power <= 0:
        raise ValueError("Invalid signal or noise power!")

    target_noise_power = signal_power / (10 ** (snr_db / 10))
    scale = np.sqrt(target_noise_power / noise_power)

    return clean_signal + noise_signal * scale

In [ ]:
rng = np.random.default_rng(2027)

X_val_noisy = {}
y_val_fixed = y_val.copy()

for noise_name in ["bw", "ma"]:
    for snr_db in VAL_SNRS:

        noisy_samples = []

        for beat in X_val:

            start = rng.integers(
                VAL_START,
                VAL_END - BEAT_LENGTH + 1
            )

            noise_segment = noise_signals[noise_name][
                start:start + BEAT_LENGTH
            ]

            noisy_beat = add_validation_noise(
                beat.squeeze(),
                noise_segment,
                snr_db
            )

            noisy_samples.append(noisy_beat)

        X_val_noisy[(noise_name, snr_db)] = np.asarray(
            noisy_samples, dtype=np.float32
        )[..., np.newaxis]

        print(
            noise_name,
            snr_db,
            "dB:",
            X_val_noisy[(noise_name, snr_db)].shape
        )

print("FIXED VALIDATION DATASET GENERATED!")

In [ ]:
for key, data in X_val_noisy.items():
    assert data.shape == (len(X_val), 288, 1)
    assert np.isfinite(data).all()

assert len(y_val_fixed) == len(X_val)

print("Validation samples:", len(y_val_fixed))
print("Noise configurations:", len(X_val_noisy))
print("All validation data finite:", True)
print("VALIDATION DATASET CHECK PASSED!")

In [ ]:
import time

sample_x, sample_y = next(iter(train_dataset_b3))

start_time = time.time()

loss, ce, cam = train_b3_step(sample_x, sample_y)

elapsed = time.time() - start_time

print("Batch size:", sample_x.shape[0])
print("Batch training time:", round(elapsed, 2), "seconds")
print("Total loss:", float(loss.numpy()))
print("CE loss:", float(ce.numpy()))
print("CAM loss:", float(cam.numpy()))

print("CPU SPEED TEST COMPLETED!")

In [ ]:
# Restore B3 from original B2 weights
b3_model.set_weights(b2_model.get_weights())

os.makedirs(B3_DIR, exist_ok=True)

initial_b3_path = os.path.join(
    B3_DIR,
    "initial_B3_from_B2.keras"
)

b3_model.save(initial_b3_path)

print("Initial B3 checkpoint saved!")
print("Path:", initial_b3_path)

same_weights = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b3_model.get_weights()
    )
)

print("B3 restored correctly:", same_weights)

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

print("Available GPUs:", gpus)

if gpus:
    print("GPU AVAILABLE!")
    print("GPU Name:", tf.config.experimental.get_device_details(gpus[0]))
else:
    print("NO GPU DETECTED!")

In [ ]:
!nvidia-smi

In [ ]:
# (Colab Drive import removed - local execution)

# (Colab Drive mount removed - local execution)

In [ ]:
import os

BASE_DIR = str(PROJECT_ROOT)

print("Project exists:", os.path.exists(BASE_DIR))

MODEL_DIR = os.path.join(BASE_DIR, "models")

B2_PATH = os.path.join(
    MODEL_DIR,
    "B2_Noise_Augmented_ResNet",
    "best_B2_resnet.keras"
)

print("B2 model exists:", os.path.exists(B2_PATH))

print("\nB3 directory exists:",
      os.path.exists(os.path.join(MODEL_DIR, "B3_CAM_Consistency")))

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import wfdb

from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

BASE_DIR = str(PROJECT_ROOT)

PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
MODEL_DIR = os.path.join(BASE_DIR, "models")

B2_DIR = os.path.join(MODEL_DIR, "B2_Noise_Augmented_ResNet")
B3_DIR = os.path.join(MODEL_DIR, "B3_CAM_Consistency")
B4_DIR = os.path.join(MODEL_DIR, "B4_Gated_CAM_Consistency")

os.makedirs(B3_DIR, exist_ok=True)
os.makedirs(B4_DIR, exist_ok=True)

print("Project paths initialized!")

In [ ]:
# (pip install moved to requirements.txt; run: pip install -r requirements.txt)

In [ ]:
import wfdb
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import json
import time

from sklearn.utils.class_weight import compute_class_weight

print("WFDB version:", wfdb.__version__)
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

print("ALL LIBRARIES IMPORTED SUCCESSFULLY!")

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import wfdb

from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

BASE_DIR = str(PROJECT_ROOT)

PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
MODEL_DIR = os.path.join(BASE_DIR, "models")

B2_DIR = os.path.join(MODEL_DIR, "B2_Noise_Augmented_ResNet")
B3_DIR = os.path.join(MODEL_DIR, "B3_CAM_Consistency")
B4_DIR = os.path.join(MODEL_DIR, "B4_Gated_CAM_Consistency")

os.makedirs(B3_DIR, exist_ok=True)
os.makedirs(B4_DIR, exist_ok=True)

print("Project paths initialized!")

In [ ]:
B2_PATH = os.path.join(
    B2_DIR,
    "best_B2_resnet.keras"
)

b2_model = tf.keras.models.load_model(
    B2_PATH,
    compile=False
)

print("B2 model loaded successfully!")
print("Input shape:", b2_model.input_shape)
print("Output shape:", b2_model.output_shape)
print("Total parameters:", b2_model.count_params())

In [ ]:
train_data = np.load(
    os.path.join(PROCESSED_DIR, "train_beats.npz")
)

X_train = train_data["X"].astype(np.float32)
y_train = train_data["y"].astype(np.int32)

val_data = np.load(
    os.path.join(PROCESSED_DIR, "validation_beats.npz")
)

X_val = val_data["X"].astype(np.float32)
y_val = val_data["y"].astype(np.int32)

if X_train.ndim == 2:
    X_train = X_train[..., np.newaxis]

if X_val.ndim == 2:
    X_val = X_val[..., np.newaxis]

print("Training ECG:", X_train.shape)
print("Training labels:", y_train.shape)

print("Validation ECG:", X_val.shape)
print("Validation labels:", y_val.shape)

print("DATA LOADING PASSED!")

In [ ]:
NSTDB_DIR = os.path.join(
    BASE_DIR,
    "data/raw/nstdb/mit-bih-noise-stress-test-database-1.0.0"
)

noise_signals = {}

for noise_name in ["bw", "ma"]:
    signal, fields = wfdb.rdsamp(
        os.path.join(NSTDB_DIR, noise_name)
    )

    noise_signals[noise_name] = signal[:, 0].astype(np.float32)

    print(
        noise_name.upper(),
        "Shape:",
        noise_signals[noise_name].shape,
        "Sampling frequency:",
        fields["fs"]
    )

noise_bank = tf.constant(
    np.stack([
        noise_signals["bw"],
        noise_signals["ma"]
    ]),
    dtype=tf.float32
)

print("Noise bank shape:", noise_bank.shape)
print("NOISE LOADING PASSED!")

In [ ]:
# Clone B2 architecture
b3_model = tf.keras.models.clone_model(b2_model)
b3_model.set_weights(b2_model.get_weights())

b4_model = tf.keras.models.clone_model(b2_model)
b4_model.set_weights(b2_model.get_weights())

print("B3 parameters:", b3_model.count_params())
print("B4 parameters:", b4_model.count_params())

print(
    "B2 and B3 identical:",
    all(np.array_equal(a, b) for a, b in
        zip(b2_model.get_weights(), b3_model.get_weights()))
)

print(
    "B2 and B4 identical:",
    all(np.array_equal(a, b) for a, b in
        zip(b2_model.get_weights(), b4_model.get_weights()))
)

print("B3 AND B4 INITIALIZATION PASSED!")

In [ ]:
# Check final activation layer
print("CAM layer:", b2_model.get_layer("activation_6").name)
print("CAM layer output:", b2_model.get_layer("activation_6").output.shape)


def create_cam_components(model):

    feature_extractor = tf.keras.Model(
        inputs=model.inputs,
        outputs=model.get_layer("activation_6").output
    )

    classifier_head = tf.keras.Model(
        inputs=model.get_layer("activation_6").output,
        outputs=model.output
    )

    return feature_extractor, classifier_head


b3_feature_extractor, b3_classifier_head = create_cam_components(
    b3_model
)

b4_feature_extractor, b4_classifier_head = create_cam_components(
    b4_model
)

print("B3 feature shape:",
      b3_feature_extractor.output_shape)

print("B4 feature shape:",
      b4_feature_extractor.output_shape)

print("CAM COMPONENTS CREATED SUCCESSFULLY!")

In [ ]:
def differentiable_cam_for_model(
    x,
    labels,
    feature_extractor,
    classifier_head
):

    with tf.GradientTape() as inner_tape:

        features = feature_extractor(x, training=False)

        inner_tape.watch(features)

        predictions = classifier_head(
            features,
            training=False
        )

        class_scores = tf.gather(
            predictions,
            labels,
            axis=1,
            batch_dims=1
        )

        total_score = tf.reduce_sum(class_scores)

    gradients = inner_tape.gradient(
        total_score,
        features
    )

    if gradients is None:
        raise ValueError("CAM gradients are None!")

    weights = tf.reduce_mean(
        gradients,
        axis=1,
        keepdims=True
    )

    cam = tf.reduce_sum(
        features * weights,
        axis=-1
    )

    cam = tf.nn.relu(cam)

    cam = cam / (
        tf.reduce_max(cam, axis=1, keepdims=True) + 1e-8
    )

    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1),
        method="bilinear"
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam, predictions


print("DIFFERENTIABLE CAM FUNCTION CREATED!")

In [ ]:
FS = 360
BEAT_LENGTH = 288

TRAIN_START = 0
TRAIN_END = 390000

VAL_START = 390000
VAL_END = 520000

TRAIN_SNRS = [24, 18, 12, 6]
VAL_SNRS = [18, 12, 6]

BATCH_SIZE = 8
LEARNING_RATE = 0.0001
LAMBDA_CAM = 0.1

print("Training SNRs:", TRAIN_SNRS)
print("Validation SNRs:", VAL_SNRS)
print("Training noise region:", TRAIN_START, TRAIN_END)
print("Configuration ready!")

In [ ]:
def add_random_noise_with_snr(ecg):

    ecg = tf.cast(ecg, tf.float32)
    ecg = tf.squeeze(ecg, axis=-1)

    noise_index = tf.random.uniform(
        [], 0, 2, dtype=tf.int32
    )

    start = tf.random.uniform(
        [],
        TRAIN_START,
        TRAIN_END - BEAT_LENGTH + 1,
        dtype=tf.int32
    )

    indices = tf.range(start, start + BEAT_LENGTH)

    noise = tf.gather(
        noise_bank[noise_index],
        indices
    )

    noise = noise - tf.reduce_mean(noise)

    snr_index = tf.random.uniform(
        [], 0, len(TRAIN_SNRS), dtype=tf.int32
    )

    snr_db = tf.gather(
        tf.constant(TRAIN_SNRS, dtype=tf.float32),
        snr_index
    )

    signal_power = tf.reduce_mean(tf.square(ecg))
    noise_power = tf.reduce_mean(tf.square(noise))

    target_noise_power = signal_power / tf.pow(
        10.0, snr_db / 10.0
    )

    scale = tf.sqrt(
        target_noise_power /
        tf.maximum(noise_power, 1e-12)
    )

    noisy_ecg = ecg + noise * scale

    return tf.expand_dims(noisy_ecg, axis=-1), snr_db


print("Noise augmentation function created!")

In [ ]:
def noise_severity_gate(snr_db):

    severity = (24.0 - snr_db) / 18.0

    return tf.clip_by_value(
        severity, 0.0, 1.0
    )


for snr in [24, 18, 12, 6]:
    print(
        f"SNR {snr} dB -> Gate:",
        float(noise_severity_gate(
            tf.constant(snr, dtype=tf.float32)
        ).numpy())
    )

print("SEVERITY GATE PASSED!")

In [ ]:
classes = np.array([0, 1, 2, 3])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_tensor = tf.constant(
    class_weights_array,
    dtype=tf.float32
)

print("Class weights:", class_weights_array)
print("CLASS WEIGHTS READY!")

In [ ]:
LAMBDA_CAM = 0.1

def weighted_ce_loss(labels, predictions):

    ce = tf.keras.losses.sparse_categorical_crossentropy(
        labels, predictions
    )

    sample_weights = tf.gather(
        class_weights_tensor, labels
    )

    return tf.reduce_mean(ce * sample_weights)


def calculate_b3_loss(
    labels,
    clean_predictions,
    noisy_predictions,
    clean_cam,
    noisy_cam
):

    clean_ce = weighted_ce_loss(
        labels, clean_predictions
    )

    noisy_ce = weighted_ce_loss(
        labels, noisy_predictions
    )

    ce_loss = (clean_ce + noisy_ce) / 2.0

    per_sample_cam_loss = tf.reduce_mean(
        tf.square(clean_cam - noisy_cam),
        axis=1
    )

    cam_loss = tf.reduce_mean(per_sample_cam_loss)

    total_loss = ce_loss + LAMBDA_CAM * cam_loss

    return total_loss, ce_loss, cam_loss


print("B3 LOSS FUNCTION READY!")

In [ ]:
def calculate_b4_loss(
    labels,
    clean_predictions,
    noisy_predictions,
    clean_cam,
    noisy_cam,
    snr_values
):

    clean_ce = weighted_ce_loss(
        labels, clean_predictions
    )

    noisy_ce = weighted_ce_loss(
        labels, noisy_predictions
    )

    ce_loss = (clean_ce + noisy_ce) / 2.0

    per_sample_cam_loss = tf.reduce_mean(
        tf.square(clean_cam - noisy_cam),
        axis=1
    )

    gates = noise_severity_gate(snr_values)

    cam_loss = tf.reduce_mean(
        per_sample_cam_loss * gates
    )

    total_loss = ce_loss + LAMBDA_CAM * cam_loss

    return total_loss, ce_loss, cam_loss


print("B4 GATED LOSS FUNCTION READY!")

In [ ]:
b3_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

b4_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

print("B3 optimizer:", b3_optimizer.name)
print("B4 optimizer:", b4_optimizer.name)

print("Learning rate:", LEARNING_RATE)
print("Lambda CAM:", LAMBDA_CAM)

print("B3 AND B4 OPTIMIZERS READY!")

In [ ]:
b3_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b3_model.get_weights()
    )
)

b4_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b4_model.get_weights()
    )
)

print("B2 and B3 identical:", b3_same)
print("B2 and B4 identical:", b4_same)

assert b3_same and b4_same

print("MODEL INITIALIZATION VERIFIED!")

In [ ]:
def train_b3_step(clean_x, labels):

    noisy_x, snr_values = tf.map_fn(
        lambda x: add_random_noise_with_snr(x),
        clean_x,
        fn_output_signature=(
            tf.TensorSpec(shape=(288, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )

    with tf.GradientTape() as outer_tape:

        clean_cam, clean_predictions = differentiable_cam_for_model(
            clean_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        noisy_cam, noisy_predictions = differentiable_cam_for_model(
            noisy_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        total_loss, ce_loss, cam_loss = calculate_b3_loss(
            labels,
            clean_predictions,
            noisy_predictions,
            clean_cam,
            noisy_cam
        )

    gradients = outer_tape.gradient(
        total_loss,
        b3_model.trainable_variables
    )

    gradient_pairs = [
        (g, v)
        for g, v in zip(
            gradients,
            b3_model.trainable_variables
        )
        if g is not None
    ]

    b3_optimizer.apply_gradients(gradient_pairs)

    return total_loss, ce_loss, cam_loss


print("B3 TRAINING FUNCTION READY!")

In [ ]:
def train_b4_step(clean_x, labels):

    noisy_x, snr_values = tf.map_fn(
        lambda x: add_random_noise_with_snr(x),
        clean_x,
        fn_output_signature=(
            tf.TensorSpec(shape=(288, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )

    with tf.GradientTape() as outer_tape:

        clean_cam, clean_predictions = differentiable_cam_for_model(
            clean_x,
            labels,
            b4_feature_extractor,
            b4_classifier_head
        )

        noisy_cam, noisy_predictions = differentiable_cam_for_model(
            noisy_x,
            labels,
            b4_feature_extractor,
            b4_classifier_head
        )

        total_loss, ce_loss, cam_loss = calculate_b4_loss(
            labels,
            clean_predictions,
            noisy_predictions,
            clean_cam,
            noisy_cam,
            snr_values
        )

    gradients = outer_tape.gradient(
        total_loss,
        b4_model.trainable_variables
    )

    gradient_pairs = [
        (g, v)
        for g, v in zip(
            gradients,
            b4_model.trainable_variables
        )
        if g is not None
    ]

    b4_optimizer.apply_gradients(gradient_pairs)

    return total_loss, ce_loss, cam_loss


print("B4 TRAINING FUNCTION READY!")

In [ ]:
BATCH_SIZE = 8

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(10000, seed=2027)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(
    "Training batches:",
    tf.data.experimental.cardinality(train_dataset).numpy()
)

print("Batch size:", BATCH_SIZE)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("GPU TRAINING DATASET READY!")

In [ ]:
import time

sample_x, sample_y = next(iter(train_dataset))

start_time = time.time()

loss, ce, cam = train_b3_step(sample_x, sample_y)

# Ensure GPU operations are completed
_ = loss.numpy()

elapsed = time.time() - start_time

print("Device: GPU")
print("Batch shape:", sample_x.shape)
print("Training time:", round(elapsed, 2), "seconds")
print("Total loss:", float(loss.numpy()))
print("CE loss:", float(ce.numpy()))
print("CAM loss:", float(cam.numpy()))

assert np.isfinite(loss.numpy())
assert np.isfinite(ce.numpy())
assert np.isfinite(cam.numpy())

print("B3 GPU ONE-BATCH TRAINING PASSED!")

In [ ]:
start_time = time.time()

loss, ce, cam = train_b4_step(sample_x, sample_y)

_ = loss.numpy()

elapsed = time.time() - start_time

print("Device: GPU")
print("Batch shape:", sample_x.shape)
print("Training time:", round(elapsed, 2), "seconds")
print("Total loss:", float(loss.numpy()))
print("CE loss:", float(ce.numpy()))
print("Gated CAM loss:", float(cam.numpy()))

assert np.isfinite(loss.numpy())
assert np.isfinite(ce.numpy())
assert np.isfinite(cam.numpy())

print("B4 GPU ONE-BATCH TRAINING PASSED!")

In [ ]:
# Restore both models from B2
b3_model.set_weights(b2_model.get_weights())
b4_model.set_weights(b2_model.get_weights())

# Reset optimizers
b3_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

b4_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

# Verify
b3_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b3_model.get_weights()
    )
)

b4_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b4_model.get_weights()
    )
)

print("B3 restored:", b3_same)
print("B4 restored:", b4_same)
print("B3 optimizer reset!")
print("B4 optimizer reset!")

assert b3_same and b4_same

In [ ]:
b3_initial_path = os.path.join(
    B3_DIR, "initial_B3_from_B2.keras"
)

b4_initial_path = os.path.join(
    B4_DIR, "initial_B4_from_B2.keras"
)

b3_model.save(b3_initial_path)
b4_model.save(b4_initial_path)

print("B3 initial checkpoint saved!")
print("B4 initial checkpoint saved!")

print(
    "B3 file exists:",
    os.path.exists(b3_initial_path)
)

print(
    "B4 file exists:",
    os.path.exists(b4_initial_path)
)

In [ ]:
def benchmark_training(train_function, dataset, model, model_name):

    batches = list(dataset.take(10))

    start_time = time.time()

    for batch_x, batch_y in batches:
        loss, ce, cam = train_function(batch_x, batch_y)
        _ = loss.numpy()

    elapsed = time.time() - start_time

    print(model_name, "10-batch time:", round(elapsed, 2), "seconds")
    print(model_name, "Average batch time:", round(elapsed / 10, 2), "seconds")


benchmark_training(
    train_b3_step,
    train_dataset,
    b3_model,
    "B3"
)

benchmark_training(
    train_b4_step,
    train_dataset,
    b4_model,
    "B4"
)

In [ ]:
# Restore original B2 weights
b3_model.set_weights(b2_model.get_weights())
b4_model.set_weights(b2_model.get_weights())

# Reset optimizers
b3_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

b4_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

# Verify
b3_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b3_model.get_weights()
    )
)

b4_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b4_model.get_weights()
    )
)

print("B3 restored:", b3_same)
print("B4 restored:", b4_same)

assert b3_same and b4_same

print("B3 AND B4 RESET COMPLETED!")

In [ ]:
rng = np.random.default_rng(2027)

X_val_noisy = {}

for noise_name in ["bw", "ma"]:
    for snr_db in VAL_SNRS:

        noisy_samples = []

        for beat in X_val:

            start = rng.integers(
                VAL_START,
                VAL_END - BEAT_LENGTH + 1
            )

            noise_segment = noise_signals[noise_name][
                start:start + BEAT_LENGTH
            ]

            noisy_beat = add_validation_noise(
                beat.squeeze(),
                noise_segment,
                snr_db
            )

            noisy_samples.append(noisy_beat)

        X_val_noisy[(noise_name, snr_db)] = np.asarray(
            noisy_samples,
            dtype=np.float32
        )[..., np.newaxis]

        print(
            noise_name,
            snr_db,
            "dB:",
            X_val_noisy[(noise_name, snr_db)].shape
        )

print("FIXED VALIDATION DATASET RECREATED!")

In [ ]:
def add_validation_noise(clean_signal, noise_segment, snr_db):

    clean_signal = np.asarray(clean_signal, dtype=np.float32)
    noise_segment = np.asarray(noise_segment, dtype=np.float32)

    noise_segment = noise_segment - np.mean(noise_segment)

    signal_power = np.mean(clean_signal ** 2)
    noise_power = np.mean(noise_segment ** 2)

    if signal_power <= 0 or noise_power <= 0:
        raise ValueError("Signal or noise power is zero!")

    target_noise_power = signal_power / (10 ** (snr_db / 10))

    scale = np.sqrt(target_noise_power / noise_power)

    noisy_signal = clean_signal + noise_segment * scale

    return noisy_signal


print("Validation noise function created!")

In [ ]:
rng = np.random.default_rng(2027)

X_val_noisy = {}

for noise_name in ["bw", "ma"]:
    for snr_db in VAL_SNRS:

        noisy_samples = []

        for beat in X_val:

            start = rng.integers(
                VAL_START,
                VAL_END - BEAT_LENGTH + 1
            )

            noise_segment = noise_signals[noise_name][
                start:start + BEAT_LENGTH
            ]

            noisy_beat = add_validation_noise(
                beat.squeeze(),
                noise_segment,
                snr_db
            )

            noisy_samples.append(noisy_beat)

        X_val_noisy[(noise_name, snr_db)] = np.asarray(
            noisy_samples,
            dtype=np.float32
        )[..., np.newaxis]

        print(
            noise_name,
            snr_db,
            "dB:",
            X_val_noisy[(noise_name, snr_db)].shape
        )

print("FIXED VALIDATION DATASET RECREATED!")

In [ ]:
for key, data in X_val_noisy.items():

    assert data.shape == (len(X_val), 288, 1)
    assert np.isfinite(data).all()

print("Validation samples:", len(X_val))
print("Noise configurations:", len(X_val_noisy))
print("Validation labels:", y_val.shape)

print("VALIDATION DATA VERIFIED!")

In [ ]:
b3_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b3_model.get_weights()
    )
)

b4_same = all(
    np.array_equal(a, b)
    for a, b in zip(
        b2_model.get_weights(),
        b4_model.get_weights()
    )
)

print("B3 restored from B2:", b3_same)
print("B4 restored from B2:", b4_same)

assert b3_same and b4_same

print("INITIAL WEIGHTS VERIFICATION PASSED!")

In [ ]:
EPOCHS = 10
BATCH_SIZE = 8
PATIENCE = 3

B3_CHECKPOINT_PATH = os.path.join(
    B3_DIR, "best_B3_resnet.keras"
)

B4_CHECKPOINT_PATH = os.path.join(
    B4_DIR, "best_B4_resnet.keras"
)

B3_HISTORY_PATH = os.path.join(
    REPORT_DIR, "B3_training_history.csv"
)

B4_HISTORY_PATH = os.path.join(
    REPORT_DIR, "B4_training_history.csv"
)

print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Early stopping patience:", PATIENCE)
print("B3 checkpoint:", B3_CHECKPOINT_PATH)
print("B4 checkpoint:", B4_CHECKPOINT_PATH)

print("TRAINING CONFIGURATION READY!")

In [ ]:
from sklearn.metrics import f1_score
import pandas as pd
import time

def train_one_epoch(train_function, dataset, model, model_name):

    total_losses = []
    ce_losses = []
    cam_losses = []

    start_time = time.time()

    for step, (batch_x, batch_y) in enumerate(dataset):

        total, ce, cam = train_function(batch_x, batch_y)

        total_losses.append(float(total.numpy()))
        ce_losses.append(float(ce.numpy()))
        cam_losses.append(float(cam.numpy()))

        if (step + 1) % 1000 == 0:
            print(
                f"{model_name} | Batch {step+1}/{len(dataset)} "
                f"| Loss: {np.mean(total_losses):.4f}"
            )

    elapsed = time.time() - start_time

    return {
        "train_loss": np.mean(total_losses),
        "train_ce_loss": np.mean(ce_losses),
        "train_cam_loss": np.mean(cam_losses),
        "training_time_min": elapsed / 60
    }


def evaluate_noisy_validation(model):

    all_f1 = []

    for (noise_name, snr_db), noisy_data in X_val_noisy.items():

        predictions = model.predict(
            noisy_data,
            batch_size=128,
            verbose=0
        )

        predicted_labels = np.argmax(predictions, axis=1)

        macro_f1 = f1_score(
            y_val,
            predicted_labels,
            average="macro",
            zero_division=0
        )

        all_f1.append(macro_f1)

        print(
            f"{noise_name.upper()} | {snr_db} dB "
            f"| Macro-F1: {macro_f1:.4f}"
        )

    return float(np.mean(all_f1))


print("Training and validation functions created!")

In [ ]:
b3_history = []

epoch = 1

print("Starting B3 Epoch", epoch)

train_result = train_one_epoch(
    train_b3_step,
    train_dataset,
    b3_model,
    "B3"
)

print("\nEvaluating B3...")

val_macro_f1 = evaluate_noisy_validation(b3_model)

epoch_result = {
    "epoch": epoch,
    **train_result,
    "val_macro_f1": val_macro_f1
}

b3_history.append(epoch_result)

# Save checkpoint
b3_model.save(B3_CHECKPOINT_PATH)

# Save history
pd.DataFrame(b3_history).to_csv(
    B3_HISTORY_PATH,
    index=False
)

print("\nB3 EPOCH 1 COMPLETED!")
print(epoch_result)
print("Checkpoint saved:", os.path.exists(B3_CHECKPOINT_PATH))

In [ ]:
# Current best score from Epoch 1
best_b3_macro_f1 = b3_history[0]["val_macro_f1"]

# Separate latest checkpoint
B3_LATEST_PATH = os.path.join(
    B3_DIR, "latest_B3_resnet.keras"
)

# Save current best model (Epoch 1)
b3_model.save(B3_CHECKPOINT_PATH)

print("Best B3 Macro-F1:", best_b3_macro_f1)
print("Epoch 1 checkpoint preserved!")

In [ ]:
epoch = 2

print("Starting B3 Epoch", epoch)

train_result = train_one_epoch(
    train_b3_step,
    train_dataset,
    b3_model,
    "B3"
)

print("\nEvaluating B3...")

val_macro_f1 = evaluate_noisy_validation(b3_model)

epoch_result = {
    "epoch": epoch,
    **train_result,
    "val_macro_f1": val_macro_f1
}

b3_history.append(epoch_result)

# Always save latest model
b3_model.save(B3_LATEST_PATH)

# Save best model only if improved
if val_macro_f1 > best_b3_macro_f1:

    best_b3_macro_f1 = val_macro_f1
    b3_model.save(B3_CHECKPOINT_PATH)

    print("NEW BEST B3 MODEL SAVED!")

# Update history
pd.DataFrame(b3_history).to_csv(
    B3_HISTORY_PATH,
    index=False
)

print("\nB3 EPOCH 2 COMPLETED!")
print(epoch_result)
print("Best Macro-F1:", best_b3_macro_f1)
print("Latest checkpoint:", os.path.exists(B3_LATEST_PATH))
print("Best checkpoint:", os.path.exists(B3_CHECKPOINT_PATH))

In [ ]:
epoch = 3

print("Starting B3 Epoch", epoch)

train_result = train_one_epoch(
    train_b3_step,
    train_dataset,
    b3_model,
    "B3"
)

print("\nEvaluating B3...")

val_macro_f1 = evaluate_noisy_validation(b3_model)

epoch_result = {
    "epoch": epoch,
    **train_result,
    "val_macro_f1": val_macro_f1
}

b3_history.append(epoch_result)

# Save latest model
b3_model.save(B3_LATEST_PATH)

# Save best model if improved
if val_macro_f1 > best_b3_macro_f1:

    best_b3_macro_f1 = val_macro_f1
    b3_model.save(B3_CHECKPOINT_PATH)

    print("NEW BEST B3 MODEL SAVED!")

# Save history
pd.DataFrame(b3_history).to_csv(
    B3_HISTORY_PATH,
    index=False
)

print("\nB3 EPOCH 3 COMPLETED!")
print(epoch_result)
print("Best Macro-F1:", best_b3_macro_f1)
print("Latest checkpoint:", os.path.exists(B3_LATEST_PATH))
print("Best checkpoint:", os.path.exists(B3_CHECKPOINT_PATH))

In [ ]:
epoch = 4

print("Starting B3 Epoch", epoch)

train_result = train_one_epoch(
    train_b3_step,
    train_dataset,
    b3_model,
    "B3"
)

print("\nEvaluating B3...")

val_macro_f1 = evaluate_noisy_validation(b3_model)

epoch_result = {
    "epoch": epoch,
    **train_result,
    "val_macro_f1": val_macro_f1
}

b3_history.append(epoch_result)

# Save latest model
b3_model.save(B3_LATEST_PATH)

# Save best model if improved
if val_macro_f1 > best_b3_macro_f1:

    best_b3_macro_f1 = val_macro_f1
    b3_model.save(B3_CHECKPOINT_PATH)

    print("NEW BEST B3 MODEL SAVED!")

# Save history
pd.DataFrame(b3_history).to_csv(
    B3_HISTORY_PATH,
    index=False
)

print("\nB3 EPOCH 4 COMPLETED!")
print(epoch_result)
print("Best Macro-F1:", best_b3_macro_f1)
print("Latest checkpoint:", os.path.exists(B3_LATEST_PATH))
print("Best checkpoint:", os.path.exists(B3_CHECKPOINT_PATH))

In [ ]:
required_items = [
    "b3_model",
    "train_b3_step",
    "train_dataset",
    "train_one_epoch",
    "b3_history",
    "best_b3_macro_f1"
]

for item in required_items:
    print(item, ":", item in globals())

In [ ]:
# Mount Google Drive
# (Colab Drive import removed - local execution)
# (Colab Drive mount removed - local execution)

# Imports
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import f1_score

# Project paths
BASE_DIR = str(PROJECT_ROOT)

PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
MODEL_DIR = os.path.join(BASE_DIR, "models")

B2_DIR = os.path.join(MODEL_DIR, "B2_Noise_Augmented_ResNet")
B3_DIR = os.path.join(MODEL_DIR, "B3_CAM_Consistency")
B4_DIR = os.path.join(MODEL_DIR, "B4_Gated_CAM_Consistency")

B2_PATH = os.path.join(B2_DIR, "best_B2_resnet.keras")

B3_CHECKPOINT_PATH = os.path.join(
    B3_DIR, "best_B3_resnet.keras"
)

B3_LATEST_PATH = os.path.join(
    B3_DIR, "latest_B3_resnet.keras"
)

B3_HISTORY_PATH = os.path.join(
    REPORT_DIR, "B3_training_history.csv"
)

# Check saved files
for path in [
    B2_PATH,
    B3_CHECKPOINT_PATH,
    B3_LATEST_PATH,
    B3_HISTORY_PATH
]:
    print(os.path.basename(path), ":", os.path.exists(path))

print("\nTensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

print("\nRECOVERY STEP 1 COMPLETED!")

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU AVAILABLE!")
    print("GPU Name:",
          tf.config.experimental.get_device_details(gpus[0]))
else:
    print("GPU NOT AVAILABLE!")

In [1]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.21.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# (Colab Drive import removed - local execution)
import os

# (Colab Drive mount removed - local execution)

PROJECT_DIR = str(PROJECT_ROOT)

print("Project Exists:", os.path.exists(PROJECT_DIR))

B3_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "B3_Attribution_Consistency"
)

print("\nB3 Directory Exists:", os.path.exists(B3_DIR))

if os.path.exists(B3_DIR):
    print("\nB3 Saved Files:")
    for root, dirs, files in os.walk(B3_DIR):
        for file in files:
            print(os.path.join(root, file))

Mounted at /content/gdrive_mount
Project Exists: True

B3 Directory Exists: False


In [3]:
import os

B3_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "B3_CAM_Consistency"
)

BEST_CHECKPOINT = os.path.join(
    B3_DIR,
    "best_B3_resnet.keras"
)

LATEST_CHECKPOINT = os.path.join(
    B3_DIR,
    "latest_B3_resnet.keras"
)

HISTORY_FILE = os.path.join(
    PROJECT_DIR,
    "reports",
    "B3_training_history.csv"
)

print("B3 Directory:", os.path.exists(B3_DIR))
print("Best Checkpoint:", os.path.exists(BEST_CHECKPOINT))
print("Latest Checkpoint:", os.path.exists(LATEST_CHECKPOINT))
print("Training History:", os.path.exists(HISTORY_FILE))

B3 Directory: True
Best Checkpoint: True
Latest Checkpoint: True
Training History: True


In [4]:
import pandas as pd
import os

history_df = pd.read_csv(HISTORY_FILE)

print("B3 Training History:")
display(history_df)

print("\nCompleted Epochs:")
print(history_df["epoch"].tolist())

print("\nLatest Checkpoint Size:")
print(f"{os.path.getsize(LATEST_CHECKPOINT) / (1024**2):.2f} MB")

print("\nBest Checkpoint Size:")
print(f"{os.path.getsize(BEST_CHECKPOINT) / (1024**2):.2f} MB")

B3 Training History:


,epoch,train_loss,train_ce_loss,train_cam_loss,training_time_min,val_macro_f1
0,1,0.402467,0.401468,0.009988,55.045435,0.303642
1,2,0.346454,0.345497,0.009563,55.103961,0.343436
2,3,0.327933,0.327065,0.008683,54.883796,0.364490



Completed Epochs:
[1, 2, 3]

Latest Checkpoint Size:
0.85 MB

Best Checkpoint Size:
0.85 MB


In [5]:
import tensorflow as tf
import os

print("Loading B3 Latest Checkpoint...")

b3_model = tf.keras.models.load_model(
    LATEST_CHECKPOINT,
    compile=False
)

print("\nB3 Model Restored Successfully!")

print("Model Parameters:", b3_model.count_params())
print("Input Shape:", b3_model.input_shape)
print("Output Shape:", b3_model.output_shape)

print("\nExpected Parameters: 579374")
print("Parameter Match:",
      b3_model.count_params() == 579374)

Loading B3 Latest Checkpoint...

B3 Model Restored Successfully!
Model Parameters: 194020
Input Shape: (None, 288, 1)
Output Shape: (None, 4)

Expected Parameters: 579374
Parameter Match: False


In [6]:
import os

print("SEARCHING ECG PROJECT MODELS...\n")

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        if file.endswith(('.keras', '.h5')):

            filepath = os.path.join(root, file)

            size = os.path.getsize(filepath) / (1024**2)

            print("File:", file)
            print("Path:", filepath)
            print("Size:", round(size, 3), "MB")
            print("-" * 60)

SEARCHING ECG PROJECT MODELS...

File: B0_CNN_Clean.keras
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B0_CNN_Clean.keras
Size: 0.665 MB
------------------------------------------------------------
File: B1_ResNet_Clean.keras
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B1_ResNet_Clean.keras
Size: 2.349 MB
------------------------------------------------------------
File: best_B2_resnet.keras
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B2_Noise_Augmented_ResNet/best_B2_resnet.keras
Size: 2.349 MB
------------------------------------------------------------
File: initial_B3_from_B2.keras
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/initial_B3_from_B2.keras
Size: 0.854 MB
------------------------------------------------------------
File: latest_B3_resnet.keras
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/latest_B3_resnet.keras
Size: 0.854 MB
--------------------------------------------

In [7]:
import tensorflow as tf
import os

model_files = {
    "B2": os.path.join(
        PROJECT_DIR,
        "models/B2_Noise_Augmented_ResNet/best_B2_resnet.keras"
    ),

    "Initial_B3": os.path.join(
        B3_DIR,
        "initial_B3_from_B2.keras"
    ),

    "Latest_B3": LATEST_CHECKPOINT,

    "Best_B3": BEST_CHECKPOINT,

    "Initial_B4": os.path.join(
        PROJECT_DIR,
        "models/B4_Gated_CAM_Consistency/initial_B4_from_B2.keras"
    )
}

loaded_models = {}

for name, path in model_files.items():

    print("\n" + "=" * 55)
    print("MODEL:", name)

    model = tf.keras.models.load_model(
        path,
        compile=False
    )

    loaded_models[name] = model

    print("Parameters:", model.count_params())
    print("Input:", model.input_shape)
    print("Output:", model.output_shape)

    print("\nLayer Names:")

    for layer in model.layers:
        print(layer.name, layer.output.shape)


MODEL: B2
Parameters: 194020
Input: (None, 288, 1)
Output: (None, 4)

Layer Names:
input_layer (None, 288, 1)
conv1d (None, 288, 32)
batch_normalization (None, 288, 32)
activation (None, 288, 32)
conv1d_1 (None, 288, 32)
batch_normalization_1 (None, 288, 32)
activation_1 (None, 288, 32)
conv1d_2 (None, 288, 32)
batch_normalization_2 (None, 288, 32)
add (None, 288, 32)
activation_2 (None, 288, 32)
conv1d_3 (None, 144, 64)
batch_normalization_3 (None, 144, 64)
activation_3 (None, 144, 64)
conv1d_4 (None, 144, 64)
conv1d_5 (None, 144, 64)
batch_normalization_4 (None, 144, 64)
batch_normalization_5 (None, 144, 64)
add_1 (None, 144, 64)
activation_4 (None, 144, 64)
conv1d_6 (None, 72, 128)
batch_normalization_6 (None, 72, 128)
activation_5 (None, 72, 128)
conv1d_7 (None, 72, 128)
conv1d_8 (None, 72, 128)
batch_normalization_7 (None, 72, 128)
batch_normalization_8 (None, 72, 128)
add_2 (None, 72, 128)
activation_6 (None, 72, 128)
global_average_pooling1d (None, 128)
dense (None, 128)
dropou

In [8]:
import numpy as np
import tensorflow as tf

initial_model = loaded_models["Initial_B3"]
latest_model = loaded_models["Latest_B3"]

initial_weights = initial_model.get_weights()
latest_weights = latest_model.get_weights()

print("Initial B3 weight arrays:", len(initial_weights))
print("Latest B3 weight arrays:", len(latest_weights))

same_weights = all(
    np.array_equal(a, b)
    for a, b in zip(initial_weights, latest_weights)
)

print("\nInitial and Latest Weights Identical:", same_weights)

print("\nLatest B3 Model Ready:", latest_model.count_params() == 194020)

Initial B3 weight arrays: 49
Latest B3 weight arrays: 49

Initial and Latest Weights Identical: False

Latest B3 Model Ready: True


In [9]:
import numpy as np

b2_model = loaded_models["B2"]
initial_b3 = loaded_models["Initial_B3"]

b2_weights = b2_model.get_weights()
b3_initial_weights = initial_b3.get_weights()

print("B2 Weight Arrays:", len(b2_weights))
print("Initial B3 Weight Arrays:", len(b3_initial_weights))

b2_b3_match = all(
    np.array_equal(a, b)
    for a, b in zip(b2_weights, b3_initial_weights)
)

print("\nB2 and Initial B3 Weights Identical:", b2_b3_match)

print("Initial B3 and Latest B3 Weights Identical:",
      same_weights)

B2 Weight Arrays: 49
Initial B3 Weight Arrays: 49

B2 and Initial B3 Weights Identical: True
Initial B3 and Latest B3 Weights Identical: False


In [10]:
import os

print("SEARCHING ORIGINAL NOTEBOOKS...\n")

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        if file.endswith(".ipynb"):

            print(os.path.join(root, file))

SEARCHING ORIGINAL NOTEBOOKS...

/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/00_Colab_Setup_and_Data_Smoke_Test.ipynb
/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/01_PhysioNet_Download_and_Data_Audit.ipynb
/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/02_Patient_Grouping_AAMI_Classification.ipynb
/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/03_ECG_Preprocessing_Beat_Extraction.ipynb
/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/04_ECG_Noise_Generation_and_SNR.ipynb
/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/05_CNN_ResNet_Clean_Baselines.ipynb
/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/06_Noise_Augmented_ResNet.ipynb
/content/gdrive_mount/MyDrive/ECG_QPAIN_2027/07_CAM_Consistency_Models.ipynb


In [11]:
import json
import os

NOTEBOOK_PATH = os.path.join(
    PROJECT_DIR,
    "07_CAM_Consistency_Models.ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("NOTEBOOK:", os.path.basename(NOTEBOOK_PATH))
print("Total Cells:", len(notebook["cells"]))

keywords = [
    "CAM",
    "cam_loss",
    "consistency",
    "optimizer",
    "learning_rate",
    "GradientTape",
    "train_step",
    "model.fit",
    "epochs",
    "checkpoint",
    "callback",
    "training_history"
]

for i, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(k.lower() in source.lower() for k in keywords):

        print("\n" + "=" * 80)
        print("CELL NUMBER:", i)
        print("=" * 80)
        print(source)


NOTEBOOK: 07_CAM_Consistency_Models.ipynb
Total Cells: 129

CELL NUMBER: 1
import os
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 2027

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = "/content/gdrive_mount/MyDrive/ECG_QPAIN_2027"

PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
MODEL_DIR = os.path.join(BASE_DIR, "models")

B2_DIR = os.path.join(
    MODEL_DIR,
    "B2_Noise_Augmented_ResNet"
)

B3_DIR = os.path.join(
    MODEL_DIR,
    "B3_CAM_Consistency"
)

B4_DIR = os.path.join(
    MODEL_DIR,
    "B4_Gated_CAM_Consistency"
)

os.makedirs(B3_DIR, exist_ok=True)
os.makedirs(B4_DIR, exist_ok=True)

print("Project paths initialized!")
print("Processed data:", os.path.exists(PROCESSED_DIR))
print("Reports:", os.path.exists(REPORT_DIR))
print("B2 model directory:", os.path.exists(B2_DIR))
print("B3 directory:", os.path.ex

In [12]:
# Mount Google Drive
# (Colab Drive import removed - local execution)
# (Colab Drive mount removed - local execution)

# Imports
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import f1_score

# Project paths
BASE_DIR = str(PROJECT_ROOT)

PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
MODEL_DIR = os.path.join(BASE_DIR, "models")

B2_DIR = os.path.join(MODEL_DIR, "B2_Noise_Augmented_ResNet")
B3_DIR = os.path.join(MODEL_DIR, "B3_CAM_Consistency")
B4_DIR = os.path.join(MODEL_DIR, "B4_Gated_CAM_Consistency")

B2_PATH = os.path.join(B2_DIR, "best_B2_resnet.keras")

B3_CHECKPOINT_PATH = os.path.join(
    B3_DIR, "best_B3_resnet.keras"
)

B3_LATEST_PATH = os.path.join(
    B3_DIR, "latest_B3_resnet.keras"
)

B3_HISTORY_PATH = os.path.join(
    REPORT_DIR, "B3_training_history.csv"
)

# Check saved files
for path in [
    B2_PATH,
    B3_CHECKPOINT_PATH,
    B3_LATEST_PATH,
    B3_HISTORY_PATH
]:
    print(os.path.basename(path), ":", os.path.exists(path))

print("\nTensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

print("\nRECOVERY STEP 1 COMPLETED!")

Drive already mounted at /content/gdrive_mount; to attempt to forcibly remount, call drive.mount("/content/gdrive_mount", force_remount=True).
best_B2_resnet.keras : True
best_B3_resnet.keras : True
latest_B3_resnet.keras : True
B3_training_history.csv : True

TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

RECOVERY STEP 1 COMPLETED!


In [13]:
# Load latest B3 model
b3_model = tf.keras.models.load_model(
    B3_LATEST_PATH,
    compile=False
)

# Restore training history
history_df = pd.read_csv(B3_HISTORY_PATH)

b3_history = history_df.to_dict("records")

# Restore best validation Macro-F1
best_b3_macro_f1 = float(
    history_df["val_macro_f1"].max()
)

# Create a fresh optimizer
# Previous Adam optimizer state was not saved
b3_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0001
)

# Verify restored model
print("B3 model restored successfully!")
print("Model parameters:", b3_model.count_params())
print("Input shape:", b3_model.input_shape)
print("Output shape:", b3_model.output_shape)

print("\nTraining history:")
display(history_df)

print("\nCompleted epochs:", len(b3_history))
print("Best validation Macro-F1:", best_b3_macro_f1)

print("\nB3 RESTORATION COMPLETED!")

B3 model restored successfully!
Model parameters: 194020
Input shape: (None, 288, 1)
Output shape: (None, 4)

Training history:


,epoch,train_loss,train_ce_loss,train_cam_loss,training_time_min,val_macro_f1
0,1,0.402467,0.401468,0.009988,55.045435,0.303642
1,2,0.346454,0.345497,0.009563,55.103961,0.343436
2,3,0.327933,0.327065,0.008683,54.883796,0.364490



Completed epochs: 3
Best validation Macro-F1: 0.3644898279927664

B3 RESTORATION COMPLETED!


In [14]:
# Check required B3 training variables

required_variables = [
    "X_train",
    "y_train",
    "X_val_noisy",
    "y_val",
    "class_weights_tensor",
    "add_random_noise_with_snr",
    "train_dataset",
    "train_b3_step",
    "train_one_epoch",
    "evaluate_noisy_validation"
]

print("B3 TRAINING ENVIRONMENT CHECK")
print("-" * 45)

for name in required_variables:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'MISSING'}")

print("\nDATA SHAPES")

if "X_train" in globals():
    print("X_train:", np.shape(X_train))

if "y_train" in globals():
    print("y_train:", np.shape(y_train))

if "y_val" in globals():
    print("y_val:", np.shape(y_val))

if "X_val_noisy" in globals():
    print("Validation noise configurations:", len(X_val_noisy))

if "class_weights_tensor" in globals():
    print("Class weights:", class_weights_tensor.numpy())

print("\nENVIRONMENT CHECK COMPLETED!")


B3 TRAINING ENVIRONMENT CHECK
---------------------------------------------
X_train: MISSING
y_train: MISSING
X_val_noisy: MISSING
y_val: MISSING
class_weights_tensor: MISSING
add_random_noise_with_snr: MISSING
train_dataset: MISSING
train_b3_step: MISSING
train_one_epoch: MISSING
evaluate_noisy_validation: MISSING

DATA SHAPES

ENVIRONMENT CHECK COMPLETED!


In [15]:
import os

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

print("SEARCHING PROCESSED DATA FILES")
print("=" * 50)

for root, dirs, files in os.walk(PROCESSED_DIR):
    level = root.replace(PROCESSED_DIR, "").count(os.sep)

    indent = " " * (4 * level)

    print(f"\n{indent}{os.path.basename(root)}/")

    for file in files:
        file_path = os.path.join(root, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)

        print(
            f"{indent}    {file} "
            f"({size_mb:.2f} MB)"
        )

print("\nPROCESSED DATA SEARCH COMPLETED!")


SEARCHING PROCESSED DATA FILES

processed/
    train_beats.npz (55.13 MB)
    train_metadata.csv (2.28 MB)
    validation_beats.npz (16.88 MB)
    validation_metadata.csv (0.83 MB)
    test_beats.npz (19.62 MB)
    test_metadata.csv (0.82 MB)

PROCESSED DATA SEARCH COMPLETED!


In [16]:
import os
import numpy as np
import pandas as pd

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

files = {
    "train": os.path.join(PROCESSED_DIR, "train_beats.npz"),
    "validation": os.path.join(PROCESSED_DIR, "validation_beats.npz"),
    "test": os.path.join(PROCESSED_DIR, "test_beats.npz")
}

metadata_files = {
    "train": os.path.join(PROCESSED_DIR, "train_metadata.csv"),
    "validation": os.path.join(PROCESSED_DIR, "validation_metadata.csv"),
    "test": os.path.join(PROCESSED_DIR, "test_metadata.csv")
}

print("=" * 60)
print("STEP 5: DATASET STRUCTURE INSPECTION")
print("=" * 60)

for name, path in files.items():
    print(f"\n{name.upper()} DATA")

    data = np.load(path, allow_pickle=True)

    print("Available keys:", data.files)

    for key in data.files:
        arr = data[key]

        print(
            f"Key: {key} | "
            f"Shape: {arr.shape} | "
            f"Dtype: {arr.dtype}"
        )

    data.close()

print("\n" + "=" * 60)
print("METADATA INSPECTION")
print("=" * 60)

for name, path in metadata_files.items():
    df = pd.read_csv(path)

    print(f"\n{name.upper()} METADATA")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())

    print("\nPossible label distributions:")

    for col in df.columns:
        if any(word in col.lower() for word in ["label", "class", "target"]):
            print(df[col].value_counts(dropna=False))

print("\nSTEP 5 COMPLETED!")

STEP 5: DATASET STRUCTURE INSPECTION

TRAIN DATA
Available keys: ['X', 'y']
Key: X | Shape: (59720, 288) | Dtype: float32
Key: y | Shape: (59720,) | Dtype: int64

VALIDATION DATA
Available keys: ['X', 'y']
Key: X | Shape: (19530, 288) | Dtype: float32
Key: y | Shape: (19530,) | Dtype: int64

TEST DATA
Available keys: ['X', 'y']
Key: X | Shape: (22156, 288) | Dtype: float32
Key: y | Shape: (22156,) | Dtype: int64

METADATA INSPECTION

TRAIN METADATA
Shape: (59720, 8)
Columns: ['Record', 'Patient_Group', 'Sample', 'Annotation', 'Class', 'Label', 'Split', 'Selected_Channel']

First 5 rows:


,Record,Patient_Group,Sample,Annotation,Class,Label,Split,Selected_Channel
0,101,patient_101,396,N,N,0,Train,MLII
1,101,patient_101,711,N,N,0,Train,MLII
2,101,patient_101,1032,N,N,0,Train,MLII
3,101,patient_101,1368,N,N,0,Train,MLII
4,101,patient_101,1712,N,N,0,Train,MLII



Possible label distributions:
Class
N    53119
V     4225
S     1956
F      420
Name: count, dtype: int64
Label
0    53119
2     4225
1     1956
3      420
Name: count, dtype: int64

VALIDATION METADATA
Shape: (19530, 8)
Columns: ['Record', 'Patient_Group', 'Sample', 'Annotation', 'Class', 'Label', 'Split', 'Selected_Channel']

First 5 rows:


,Record,Patient_Group,Sample,Annotation,Class,Label,Split,Selected_Channel
0,100,patient_100,370,N,N,0,Validation,MLII
1,100,patient_100,662,N,N,0,Validation,MLII
2,100,patient_100,946,N,N,0,Validation,MLII
3,100,patient_100,1231,N,N,0,Validation,MLII
4,100,patient_100,1515,N,N,0,Validation,MLII



Possible label distributions:
Class
N    17533
V     1518
S      462
F       17
Name: count, dtype: int64
Label
0    17533
2     1518
1      462
3       17
Name: count, dtype: int64

TEST METADATA
Shape: (22156, 8)
Columns: ['Record', 'Patient_Group', 'Sample', 'Annotation', 'Class', 'Label', 'Split', 'Selected_Channel']

First 5 rows:


,Record,Patient_Group,Sample,Annotation,Class,Label,Split,Selected_Channel
0,111,patient_111,197,L,N,0,Test,MLII
1,111,patient_111,489,L,N,0,Test,MLII
2,111,patient_111,804,L,N,0,Test,MLII
3,111,patient_111,1127,L,N,0,Test,MLII
4,111,patient_111,1430,L,N,0,Test,MLII



Possible label distributions:
Class
N    19936
V     1492
F      365
S      363
Name: count, dtype: int64
Label
0    19936
2     1492
3      365
1      363
Name: count, dtype: int64

STEP 5 COMPLETED!


In [17]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

# Load datasets
train_data = np.load(
    os.path.join(PROCESSED_DIR, "train_beats.npz")
)

val_data = np.load(
    os.path.join(PROCESSED_DIR, "validation_beats.npz")
)

test_data = np.load(
    os.path.join(PROCESSED_DIR, "test_beats.npz")
)

# Extract signals and labels
X_train = train_data["X"].astype(np.float32)
y_train = train_data["y"].astype(np.int64)

X_val = val_data["X"].astype(np.float32)
y_val = val_data["y"].astype(np.int64)

X_test = test_data["X"].astype(np.float32)
y_test = test_data["y"].astype(np.int64)

# Reshape signals for model input
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("=" * 55)
print("DATASET LOADING COMPLETED")
print("=" * 55)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining label distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation label distribution:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest label distribution:")
print(pd.Series(y_test).value_counts().sort_index())

assert X_train.shape[1:] == (288, 1)
assert X_val.shape[1:] == (288, 1)
assert X_test.shape[1:] == (288, 1)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

print("\nSTEP 6 COMPLETED SUCCESSFULLY!")

DATASET LOADING COMPLETED
X_train: (59720, 288, 1)
y_train: (59720,)
X_val: (19530, 288, 1)
y_val: (19530,)
X_test: (22156, 288, 1)
y_test: (22156,)

Training label distribution:
0    53119
1     1956
2     4225
3      420
Name: count, dtype: int64

Validation label distribution:
0    17533
1      462
2     1518
3       17
Name: count, dtype: int64

Test label distribution:
0    19936
1      363
2     1492
3      365
Name: count, dtype: int64

STEP 6 COMPLETED SUCCESSFULLY!


In [18]:
import numpy as np
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_tensor = tf.constant(
    class_weights,
    dtype=tf.float32
)

print("CLASS WEIGHTS")
for i, weight in enumerate(class_weights):
    print(f"Class {i}: {weight:.6f}")

print("\nTensor:", class_weights_tensor)
print("Shape:", class_weights_tensor.shape)

assert class_weights_tensor.shape == (4,)
assert np.all(np.isfinite(class_weights))

print("\nSTEP 7 COMPLETED!")

CLASS WEIGHTS
Class 0: 0.281067
Class 1: 7.632924
Class 2: 3.533728
Class 3: 35.547619

Tensor: tf.Tensor([ 0.28106704  7.6329246   3.533728   35.54762   ], shape=(4,), dtype=float32)
Shape: (4,)

STEP 7 COMPLETED!


In [19]:
BATCH_SIZE = 128
LEARNING_RATE = 0.0001
LAMBDA_CAM = 0.1

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(10000, seed=2027)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training dataset ready!")

print("Batch size:", BATCH_SIZE)
print("Total samples:", len(X_train))
print("Total batches:", len(train_dataset))

sample_x, sample_y = next(iter(train_dataset))

print("Batch X shape:", sample_x.shape)
print("Batch y shape:", sample_y.shape)

assert sample_x.shape == (128, 288, 1)
assert sample_y.shape == (128,)

print("\nSTEP 8 COMPLETED!")

Training dataset ready!
Batch size: 128
Total samples: 59720
Total batches: 467
Batch X shape: (128, 288, 1)
Batch y shape: (128,)

STEP 8 COMPLETED!


In [20]:
import os
import pandas as pd

B3_DIR = os.path.join(
    BASE_DIR,
    "models",
    "B3_CAM_Consistency"
)

REPORT_DIR = os.path.join(BASE_DIR, "reports")

B3_LATEST_PATH = os.path.join(
    B3_DIR,
    "latest_B3_resnet.keras"
)

B3_CHECKPOINT_PATH = os.path.join(
    B3_DIR,
    "best_B3_resnet.keras"
)

B3_HISTORY_PATH = os.path.join(
    REPORT_DIR,
    "B3_training_history.csv"
)

b3_history_df = pd.read_csv(B3_HISTORY_PATH)

b3_history = b3_history_df.to_dict("records")

completed_epochs = len(b3_history)

best_b3_macro_f1 = float(
    b3_history_df["val_macro_f1"].max()
)

print("Completed epochs:", completed_epochs)
print("Next epoch:", completed_epochs + 1)
print("Best validation Macro-F1:", best_b3_macro_f1)

display(b3_history_df)

assert completed_epochs == 3
assert os.path.exists(B3_LATEST_PATH)

print("\nSTEP 9 COMPLETED!")

Completed epochs: 3
Next epoch: 4
Best validation Macro-F1: 0.3644898279927664


,epoch,train_loss,train_ce_loss,train_cam_loss,training_time_min,val_macro_f1
0,1,0.402467,0.401468,0.009988,55.045435,0.303642
1,2,0.346454,0.345497,0.009563,55.103961,0.343436
2,3,0.327933,0.327065,0.008683,54.883796,0.364490



STEP 9 COMPLETED!


In [21]:
b3_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

print("B3 optimizer initialized!")
print("Learning rate:", LEARNING_RATE)

print("\nSTEP 10 COMPLETED!")

B3 optimizer initialized!
Learning rate: 0.0001

STEP 10 COMPLETED!


In [22]:
print("B3 MODEL VERIFICATION")
print("=" * 45)

print("Model parameters:", b3_model.count_params())
print("Input:", b3_model.input_shape)
print("Output:", b3_model.output_shape)

b3_feature_extractor = tf.keras.Model(
    inputs=b3_model.inputs,
    outputs=b3_model.get_layer("activation_6").output
)

b3_classifier_head = tf.keras.Model(
    inputs=b3_model.get_layer("activation_6").output,
    outputs=b3_model.output
)

print("CAM feature output:",
      b3_feature_extractor.output_shape)

print("Classifier output:",
      b3_classifier_head.output_shape)

assert b3_model.count_params() == 194020
assert b3_model.output_shape == (None, 4)

print("\nSTEP 11 COMPLETED!")

B3 MODEL VERIFICATION
Model parameters: 194020
Input: (None, 288, 1)
Output: (None, 4)
CAM feature output: (None, 72, 128)
Classifier output: (None, 4)

STEP 11 COMPLETED!


In [23]:
import os

ROOT = BASE_DIR

print("=" * 65)
print("SEARCHING ORIGINAL NOISE IMPLEMENTATION")
print("=" * 65)

keywords = [
    "04_NSTDB",
    "05_CNN",
    "06_Noise",
    "noise.py",
    "noise_protocol",
    "snr_unit_tests",
    "noise_generation_config"
]

for root, dirs, files in os.walk(ROOT):

    # Avoid searching large raw and processed data folders
    dirs[:] = [
        d for d in dirs
        if d not in ["mitdb", "nstdb", "processed", "__pycache__"]
    ]

    for filename in files:
        if any(k.lower() in filename.lower() for k in keywords):

            path = os.path.join(root, filename)

            print("\nFOUND:", filename)
            print("PATH:", path)

print("\nSEARCH COMPLETED!")

SEARCHING ORIGINAL NOISE IMPLEMENTATION

FOUND: 05_CNN_ResNet_Clean_Baselines.ipynb
PATH: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/05_CNN_ResNet_Clean_Baselines.ipynb

FOUND: 06_Noise_Augmented_ResNet.ipynb
PATH: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/06_Noise_Augmented_ResNet.ipynb

FOUND: noise_generation_config.json
PATH: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/reports/noise_generation_config.json

SEARCH COMPLETED!


In [24]:
def add_random_noise_with_snr

SyntaxError: expected '(' (1732335293.py, line 1)

In [26]:
def add_random_noise_with_snr(x):
    # Function body এখানে থাকবে
    pass

In [27]:
def add_random_noise_with_snr(x):

_IncompleteInputError: incomplete input (1658939297.py, line 1)

In [28]:
import json

notebook_path = os.path.join(
    BASE_DIR, "07_CAM_Consistency_Models.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if "add_random_noise_with_snr" in source:
        print(f"\nFOUND IN CELL {i}:\n")
        print(source)


FOUND IN CELL 41:

def add_random_noise_with_snr(ecg):

    ecg = tf.cast(ecg, tf.float32)
    ecg = tf.squeeze(ecg, axis=-1)

    # Select BW or MA
    noise_index = tf.random.uniform(
        [], 0, 2, dtype=tf.int32
    )

    # Select noise segment from training region
    start = tf.random.uniform(
        [],
        TRAIN_START,
        TRAIN_END - BEAT_LENGTH + 1,
        dtype=tf.int32
    )

    indices = tf.range(start, start + BEAT_LENGTH)

    noise = tf.gather(
        noise_bank[noise_index],
        indices
    )

    noise = noise - tf.reduce_mean(noise)

    # Select SNR
    snr_index = tf.random.uniform(
        [], 0, len(TRAIN_SNRS), dtype=tf.int32
    )

    snr_db = tf.gather(
        tf.constant(TRAIN_SNRS, dtype=tf.float32),
        snr_index
    )

    signal_power = tf.reduce_mean(tf.square(ecg))
    noise_power = tf.reduce_mean(tf.square(noise))

    target_noise_power = (
        signal_power / tf.pow(10.0, snr_db / 10.0)
    )

    scale = tf.sqrt(
      

In [29]:
def add_random_noise_with_snr(ecg):

    ecg = tf.cast(ecg, tf.float32)
    ecg = tf.squeeze(ecg, axis=-1)

    # Select BW or MA noise
    noise_index = tf.random.uniform(
        [], 0, 2, dtype=tf.int32
    )

    # Select noise segment from training region
    start = tf.random.uniform(
        [],
        TRAIN_START,
        TRAIN_END - BEAT_LENGTH + 1,
        dtype=tf.int32
    )

    indices = tf.range(start, start + BEAT_LENGTH)

    noise = tf.gather(
        noise_bank[noise_index],
        indices
    )

    noise = noise - tf.reduce_mean(noise)

    # Select SNR
    snr_index = tf.random.uniform(
        [], 0, len(TRAIN_SNRS), dtype=tf.int32
    )

    snr_db = tf.gather(
        tf.constant(TRAIN_SNRS, dtype=tf.float32),
        snr_index
    )

    signal_power = tf.reduce_mean(tf.square(ecg))
    noise_power = tf.reduce_mean(tf.square(noise))

    target_noise_power = (
        signal_power / tf.pow(10.0, snr_db / 10.0)
    )

    scale = tf.sqrt(
        target_noise_power /
        tf.maximum(noise_power, 1e-12)
    )

    noisy_ecg = ecg + noise * scale

    return (
        tf.expand_dims(noisy_ecg, axis=-1),
        snr_db
    )


print("Original B3 noise augmentation function restored!")

Original B3 noise augmentation function restored!


In [30]:
test_clean = tf.constant(X_train[:1])

test_noisy, test_snr = add_random_noise_with_snr(
    test_clean[0]
)

print("Clean shape:", test_clean.shape)
print("Noisy shape:", test_noisy.shape)
print("Selected SNR:", float(test_snr.numpy()), "dB")

assert test_noisy.shape == (288, 1)
assert float(test_snr.numpy()) in TRAIN_SNRS

print("ORIGINAL SNR-AWARE AUGMENTATION PASSED!")

NameError: name 'TRAIN_START' is not defined

In [31]:
import json
import os

notebook_path = os.path.join(
    BASE_DIR,
    "07_CAM_Consistency_Models.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

search_terms = [
    "TRAIN_START",
    "TRAIN_END",
    "TRAIN_SNRS",
    "BEAT_LENGTH",
    "noise_bank"
]

print("=" * 60)
print("SEARCHING ORIGINAL NOISE CONFIGURATION")
print("=" * 60)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    matched = [
        term for term in search_terms
        if term in source
    ]

    if matched:
        print(f"\nCELL {i} | MATCHED: {matched}")
        print("-" * 50)
        print(source[:6000])

print("\nCONFIGURATION SEARCH COMPLETED!")

SEARCHING ORIGINAL NOISE CONFIGURATION

CELL 19 | MATCHED: ['noise_bank']
--------------------------------------------------
noise_bank = tf.constant(
    np.stack([
        noise_signals["bw"],
        noise_signals["ma"]
    ]),
    dtype=tf.float32
)

print("Noise bank shape:", noise_bank.shape)

assert noise_bank.shape == (2, 650000)

print("NOISE BANK VALIDATION PASSED!")

CELL 20 | MATCHED: ['TRAIN_START', 'TRAIN_END', 'TRAIN_SNRS', 'BEAT_LENGTH', 'noise_bank']
--------------------------------------------------
FS = 360
BEAT_LENGTH = 288

TRAIN_SNRS = [24, 18, 12, 6]

TRAIN_START = 0
TRAIN_END = 390000


def add_random_noise(ecg):

    ecg = tf.cast(ecg, tf.float32)

    ecg = tf.squeeze(ecg, axis=-1)

    # Randomly select BW or MA noise
    noise_index = tf.random.uniform(
        [],
        0,
        2,
        dtype=tf.int32
    )

    # Select a random noise segment
    start = tf.random.uniform(
        [],
        TRAIN_START,
        TRAIN_END - BEAT_LENGTH + 1,
       

In [32]:
# Original Notebook 07 configuration

FS = 360
BEAT_LENGTH = 288

TRAIN_START = 0
TRAIN_END = 390000

VAL_START = 390000
VAL_END = 520000

TRAIN_SNRS = [24, 18, 12, 6]
VAL_SNRS = [18, 12, 6]

print("NOISE CONFIGURATION RESTORED")
print("=" * 45)

print("Sampling frequency:", FS)
print("Beat length:", BEAT_LENGTH)

print("Training region:", TRAIN_START, "to", TRAIN_END)
print("Validation region:", VAL_START, "to", VAL_END)

print("Training SNRs:", TRAIN_SNRS)
print("Validation SNRs:", VAL_SNRS)

print("\nCONFIGURATION RESTORATION PASSED!")

NOISE CONFIGURATION RESTORED
Sampling frequency: 360
Beat length: 288
Training region: 0 to 390000
Validation region: 390000 to 520000
Training SNRs: [24, 18, 12, 6]
Validation SNRs: [18, 12, 6]

CONFIGURATION RESTORATION PASSED!


In [33]:
print("NOISE BANK VERIFICATION")
print("=" * 45)

print("noise_signals available:", "noise_signals" in globals())
print("noise_bank available:", "noise_bank" in globals())

if "noise_bank" in globals():
    print("Noise bank shape:", noise_bank.shape)

    assert noise_bank.shape == (2, 650000)

    print("NOISE BANK VALIDATION PASSED!")

else:
    print("NOISE BANK MISSING - NEED TO RELOAD")

NOISE BANK VERIFICATION
noise_signals available: False
noise_bank available: False
NOISE BANK MISSING - NEED TO RELOAD


In [34]:
test_clean = tf.constant(X_train[:1])

test_noisy, test_snr = add_random_noise_with_snr(
    test_clean[0]
)

print("Clean shape:", test_clean.shape)
print("Noisy shape:", test_noisy.shape)
print("Selected SNR:", float(test_snr.numpy()), "dB")

assert test_noisy.shape == (288, 1)
assert float(test_snr.numpy()) in TRAIN_SNRS

print("ORIGINAL SNR-AWARE AUGMENTATION PASSED!")

NameError: name 'noise_bank' is not defined

In [35]:
import os
import numpy as np
import tensorflow as tf
import wfdb

NSTDB_DIR = os.path.join(
    BASE_DIR,
    "data/raw/nstdb/mit-bih-noise-stress-test-database-1.0.0"
)

noise_signals = {}

for noise_name in ["bw", "ma"]:

    signal, fields = wfdb.rdsamp(
        os.path.join(NSTDB_DIR, noise_name)
    )

    noise_signals[noise_name] = signal[:, 0].astype(np.float32)

    print(
        noise_name.upper(),
        "Shape:", noise_signals[noise_name].shape,
        "Sampling frequency:", fields["fs"]
    )

noise_bank = tf.constant(
    np.stack([
        noise_signals["bw"],
        noise_signals["ma"]
    ]),
    dtype=tf.float32
)

print("\nNoise bank shape:", noise_bank.shape)

assert noise_bank.shape == (2, 650000)

print("NOISE BANK LOADING PASSED!")

ModuleNotFoundError: No module named 'wfdb'

In [36]:
# (pip install moved to requirements.txt; run: pip install -r requirements.txt)

In [37]:
import wfdb

print("WFDB version:", wfdb.__version__)
print("WFDB IMPORT SUCCESSFUL!")

WFDB version: 4.3.1
WFDB IMPORT SUCCESSFUL!


In [38]:
import os
import numpy as np
import tensorflow as tf
import wfdb

NSTDB_DIR = os.path.join(
    BASE_DIR,
    "data/raw/nstdb/mit-bih-noise-stress-test-database-1.0.0"
)

noise_signals = {}

for noise_name in ["bw", "ma"]:

    signal, fields = wfdb.rdsamp(
        os.path.join(NSTDB_DIR, noise_name)
    )

    noise_signals[noise_name] = signal[:, 0].astype(np.float32)

    print(
        noise_name.upper(),
        "Shape:", noise_signals[noise_name].shape,
        "Sampling frequency:", fields["fs"]
    )

noise_bank = tf.constant(
    np.stack([
        noise_signals["bw"],
        noise_signals["ma"]
    ]),
    dtype=tf.float32
)

print("\nNoise bank shape:", noise_bank.shape)

assert noise_bank.shape == (2, 650000)

print("NOISE BANK LOADING PASSED!")

BW Shape: (650000,) Sampling frequency: 360
MA Shape: (650000,) Sampling frequency: 360

Noise bank shape: (2, 650000)
NOISE BANK LOADING PASSED!


In [39]:
test_clean = tf.constant(X_train[:1])

test_noisy, test_snr = add_random_noise_with_snr(
    test_clean[0]
)

print("Clean shape:", test_clean.shape)
print("Noisy shape:", test_noisy.shape)
print("Selected SNR:", float(test_snr.numpy()), "dB")

assert test_noisy.shape == (288, 1)
assert float(test_snr.numpy()) in TRAIN_SNRS

print("ORIGINAL SNR-AWARE AUGMENTATION PASSED!")

Clean shape: (1, 288, 1)
Noisy shape: (288, 1)
Selected SNR: 18.0 dB
ORIGINAL SNR-AWARE AUGMENTATION PASSED!


In [40]:
VAL_START = 390000
VAL_END = 520000

VAL_SNRS = [18, 12, 6]
BEAT_LENGTH = 288

print("Validation region:", VAL_START, "to", VAL_END)
print("Validation SNRs:", VAL_SNRS)
print("Validation noise length:", VAL_END - VAL_START)
print("VALIDATION CONFIGURATION READY!")

Validation region: 390000 to 520000
Validation SNRs: [18, 12, 6]
Validation noise length: 130000
VALIDATION CONFIGURATION READY!


In [41]:
import json
import os

notebook_path = os.path.join(
    BASE_DIR,
    "07_CAM_Consistency_Models.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    if "def add_validation_noise" in source:
        print(f"FOUND IN CELL {i}")
        print("=" * 50)
        print(source)

FOUND IN CELL 66
def add_validation_noise(clean_signal, noise_signal, snr_db):
    clean_signal = np.asarray(clean_signal, dtype=np.float32)
    noise_signal = np.asarray(noise_signal, dtype=np.float32)

    noise_signal = noise_signal - np.mean(noise_signal)

    signal_power = np.mean(clean_signal ** 2)
    noise_power = np.mean(noise_signal ** 2)

    if signal_power <= 0 or noise_power <= 0:
        raise ValueError("Invalid signal or noise power!")

    target_noise_power = signal_power / (10 ** (snr_db / 10))
    scale = np.sqrt(target_noise_power / noise_power)

    return clean_signal + noise_signal * scale
FOUND IN CELL 103
def add_validation_noise(clean_signal, noise_segment, snr_db):

    clean_signal = np.asarray(clean_signal, dtype=np.float32)
    noise_segment = np.asarray(noise_segment, dtype=np.float32)

    noise_segment = noise_segment - np.mean(noise_segment)

    signal_power = np.mean(clean_signal ** 2)
    noise_power = np.mean(noise_segment ** 2)

    if signal_p

In [42]:
def add_validation_noise(clean_signal, noise_segment, snr_db):

    clean_signal = np.asarray(clean_signal, dtype=np.float32)
    noise_segment = np.asarray(noise_segment, dtype=np.float32)

    noise_segment = noise_segment - np.mean(noise_segment)

    signal_power = np.mean(clean_signal ** 2)
    noise_power = np.mean(noise_segment ** 2)

    if signal_power <= 0 or noise_power <= 0:
        raise ValueError("Signal or noise power is zero!")

    target_noise_power = signal_power / (10 ** (snr_db / 10))

    scale = np.sqrt(target_noise_power / noise_power)

    noisy_signal = clean_signal + noise_segment * scale

    return noisy_signal


print("Original validation noise function restored!")


Original validation noise function restored!


In [43]:
rng = np.random.default_rng(2027)

X_val_noisy = {}

for noise_name in ["bw", "ma"]:

    for snr_db in VAL_SNRS:

        noisy_samples = []

        for beat in X_val:

            start = rng.integers(
                VAL_START,
                VAL_END - BEAT_LENGTH + 1
            )

            noise_segment = noise_signals[noise_name][
                start:start + BEAT_LENGTH
            ]

            noisy_beat = add_validation_noise(
                beat.squeeze(),
                noise_segment,
                snr_db
            )

            noisy_samples.append(noisy_beat)

        X_val_noisy[(noise_name, snr_db)] = np.asarray(
            noisy_samples,
            dtype=np.float32
        )[..., np.newaxis]

        print(
            noise_name,
            snr_db,
            "dB:",
            X_val_noisy[(noise_name, snr_db)].shape
        )

print("\nFIXED VALIDATION DATASET GENERATED!")

bw 18 dB: (19530, 288, 1)
bw 12 dB: (19530, 288, 1)
bw 6 dB: (19530, 288, 1)
ma 18 dB: (19530, 288, 1)
ma 12 dB: (19530, 288, 1)
ma 6 dB: (19530, 288, 1)

FIXED VALIDATION DATASET GENERATED!


In [44]:
print("VALIDATION DATASET VERIFICATION")
print("=" * 50)

print("Configurations:", len(X_val_noisy))

for key, data in X_val_noisy.items():
    print(key, data.shape)

assert len(X_val_noisy) == 6

for data in X_val_noisy.values():
    assert data.shape == (19530, 288, 1)

print("\nVALIDATION DATASET VERIFICATION PASSED!")

VALIDATION DATASET VERIFICATION
Configurations: 6
('bw', 18) (19530, 288, 1)
('bw', 12) (19530, 288, 1)
('bw', 6) (19530, 288, 1)
('ma', 18) (19530, 288, 1)
('ma', 12) (19530, 288, 1)
('ma', 6) (19530, 288, 1)

VALIDATION DATASET VERIFICATION PASSED!


In [45]:
required_variables = [
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_val_noisy",
    "class_weights_tensor",
    "noise_bank",
    "add_random_noise_with_snr",
    "add_validation_noise",
    "train_dataset",
    "train_b3_step",
    "train_one_epoch",
    "evaluate_noisy_validation",
    "b3_model",
    "b3_optimizer",
    "b3_history",
    "best_b3_macro_f1"
]

print("=" * 60)
print("B3 TRAINING ENVIRONMENT CHECK")
print("=" * 60)

all_available = True

for name in required_variables:
    exists = name in globals()

    print(
        f"{name}:",
        "AVAILABLE" if exists else "MISSING"
    )

    if not exists:
        all_available = False

print("\nDATA SHAPES")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_val_noisy configurations:", len(X_val_noisy))
print("Noise bank:", noise_bank.shape)

print("\nB3 MODEL")

print("Parameters:", b3_model.count_params())
print("Optimizer:", type(b3_optimizer).__name__)

print("\nTraining history:")
print(b3_history)

print("\nBest validation macro F1:", best_b3_macro_f1)

print("\nALL VARIABLES AVAILABLE:", all_available)


B3 TRAINING ENVIRONMENT CHECK
X_train: AVAILABLE
y_train: AVAILABLE
X_val: AVAILABLE
y_val: AVAILABLE
X_val_noisy: AVAILABLE
class_weights_tensor: AVAILABLE
noise_bank: AVAILABLE
add_random_noise_with_snr: AVAILABLE
add_validation_noise: AVAILABLE
train_dataset: AVAILABLE
train_b3_step: MISSING
train_one_epoch: MISSING
evaluate_noisy_validation: MISSING
b3_model: AVAILABLE
b3_optimizer: AVAILABLE
b3_history: AVAILABLE
best_b3_macro_f1: AVAILABLE

DATA SHAPES
X_train: (59720, 288, 1)
y_train: (59720,)
X_val: (19530, 288, 1)
y_val: (19530,)
X_val_noisy configurations: 6
Noise bank: (2, 650000)

B3 MODEL
Parameters: 194020
Optimizer: Adam

Training history:
[{'epoch': 1, 'train_loss': 0.4024671060703096, 'train_ce_loss': 0.4014683047475447, 'train_cam_loss': 0.009988015743035, 'training_time_min': 55.0454349676768, 'val_macro_f1': 0.3036416111401026}, {'epoch': 2, 'train_loss': 0.346453770432479, 'train_ce_loss': 0.3454974999897711, 'train_cam_loss': 0.0095627024435118, 'training_time_min

In [46]:
import json
import os

notebook_path = os.path.join(
    BASE_DIR,
    "07_CAM_Consistency_Models.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

functions_to_find = [
    "def train_b3_step",
    "def train_one_epoch",
    "def evaluate_noisy_validation"
]

print("=" * 65)
print("RECOVERING ORIGINAL B3 TRAINING FUNCTIONS")
print("=" * 65)

for function_name in functions_to_find:

    print("\n\nSEARCHING:", function_name)
    print("-" * 65)

    found = False

    for i, cell in enumerate(nb["cells"]):
        source = "".join(cell.get("source", []))

        if function_name in source:
            print(f"\nFOUND IN CELL {i}:\n")
            print(source)
            found = True

    if not found:
        print("FUNCTION NOT FOUND!")

RECOVERING ORIGINAL B3 TRAINING FUNCTIONS


SEARCHING: def train_b3_step
-----------------------------------------------------------------

FOUND IN CELL 55:

def train_b3_step(clean_x, labels):

    noisy_x, _ = tf.map_fn(
        lambda x: add_random_noise_with_snr(x),
        clean_x,
        fn_output_signature=(
            tf.TensorSpec(shape=(288, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )

    with tf.GradientTape() as outer_tape:

        clean_cam, clean_predictions = differentiable_cam_for_model(
            clean_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        noisy_cam, _ = differentiable_cam_for_model(
            noisy_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        total_loss, ce_loss, cam_loss = calculate_b3_loss(
            labels,
            clean_predictions,
            clean_cam,
            noisy

In [47]:
def train_b3_step(clean_x, labels):

    noisy_x, snr_values = tf.map_fn(
        lambda x: add_random_noise_with_snr(x),
        clean_x,
        fn_output_signature=(
            tf.TensorSpec(shape=(288, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )

    with tf.GradientTape() as outer_tape:

        clean_cam, clean_predictions = differentiable_cam_for_model(
            clean_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        noisy_cam, noisy_predictions = differentiable_cam_for_model(
            noisy_x,
            labels,
            b3_feature_extractor,
            b3_classifier_head
        )

        total_loss, ce_loss, cam_loss = calculate_b3_loss(
            labels,
            clean_predictions,
            noisy_predictions,
            clean_cam,
            noisy_cam
        )

    gradients = outer_tape.gradient(
        total_loss,
        b3_model.trainable_variables
    )

    gradient_pairs = [
        (g, v)
        for g, v in zip(
            gradients,
            b3_model.trainable_variables
        )
        if g is not None
    ]

    b3_optimizer.apply_gradients(gradient_pairs)

    return total_loss, ce_loss, cam_loss


print("B3 TRAINING FUNCTION RESTORED!")

B3 TRAINING FUNCTION RESTORED!


In [48]:
from sklearn.metrics import f1_score
import pandas as pd
import time

def train_one_epoch(train_function, dataset, model, model_name):

    total_losses = []
    ce_losses = []
    cam_losses = []

    start_time = time.time()

    for step, (batch_x, batch_y) in enumerate(dataset):

        total, ce, cam = train_function(batch_x, batch_y)

        total_losses.append(float(total.numpy()))
        ce_losses.append(float(ce.numpy()))
        cam_losses.append(float(cam.numpy()))

        if (step + 1) % 1000 == 0:
            print(
                f"{model_name} | Batch {step+1}/{len(dataset)} "
                f"| Loss: {np.mean(total_losses):.4f}"
            )

    elapsed = time.time() - start_time

    return {
        "train_loss": np.mean(total_losses),
        "train_ce_loss": np.mean(ce_losses),
        "train_cam_loss": np.mean(cam_losses),
        "training_time_min": elapsed / 60
    }


def evaluate_noisy_validation(model):

    all_f1 = []

    for (noise_name, snr_db), noisy_data in X_val_noisy.items():

        predictions = model.predict(
            noisy_data,
            batch_size=128,
            verbose=0
        )

        predicted_labels = np.argmax(predictions, axis=1)

        macro_f1 = f1_score(
            y_val,
            predicted_labels,
            average="macro",
            zero_division=0
        )

        all_f1.append(macro_f1)

        print(
            f"{noise_name.upper()} | {snr_db} dB "
            f"| Macro-F1: {macro_f1:.4f}"
        )

    return float(np.mean(all_f1))


print("TRAINING AND VALIDATION FUNCTIONS RESTORED!")

TRAINING AND VALIDATION FUNCTIONS RESTORED!


In [49]:
required_variables = [
    "train_b3_step",
    "train_one_epoch",
    "evaluate_noisy_validation",
    "differentiable_cam_for_model",
    "calculate_b3_loss",
    "b3_feature_extractor",
    "b3_classifier_head",
    "b3_model",
    "b3_optimizer",
    "train_dataset",
    "X_val_noisy",
    "class_weights_tensor"
]

print("=" * 55)
print("FINAL B3 ENVIRONMENT CHECK")
print("=" * 55)

all_available = True

for name in required_variables:

    exists = name in globals()

    print(
        name,
        ":",
        "AVAILABLE" if exists else "MISSING"
    )

    if not exists:
        all_available = False

print("\nTraining batches:", len(train_dataset))
print("Model parameters:", b3_model.count_params())
print("Training history epochs:", len(b3_history))
print("Best validation F1:", best_b3_macro_f1)

print("\nALL AVAILABLE:", all_available)

FINAL B3 ENVIRONMENT CHECK
train_b3_step : AVAILABLE
train_one_epoch : AVAILABLE
evaluate_noisy_validation : AVAILABLE
differentiable_cam_for_model : MISSING
calculate_b3_loss : MISSING
b3_feature_extractor : AVAILABLE
b3_classifier_head : AVAILABLE
b3_model : AVAILABLE
b3_optimizer : AVAILABLE
train_dataset : AVAILABLE
X_val_noisy : AVAILABLE
class_weights_tensor : AVAILABLE

Training batches: 467
Model parameters: 194020
Training history epochs: 3
Best validation F1: 0.3644898279927664

ALL AVAILABLE: False


In [50]:
# ============================================
# STEP 29: RESTORE B3 CAM AND LOSS FUNCTIONS
# ============================================

def differentiable_cam_for_model(
    x, labels, feature_extractor, classifier_head
):
    x = tf.cast(x, tf.float32)
    labels = tf.cast(labels, tf.int32)

    with tf.GradientTape() as tape:
        features = feature_extractor(x, training=False)
        tape.watch(features)

        predictions = classifier_head(features, training=False)

        indices = tf.stack(
            [
                tf.range(tf.shape(labels)[0]),
                labels
            ],
            axis=1
        )

        target_scores = tf.gather_nd(predictions, indices)
        target_score_sum = tf.reduce_sum(target_scores)

    gradients = tape.gradient(target_score_sum, features)

    pooled_gradients = tf.reduce_mean(
        gradients, axis=1, keepdims=True
    )

    cam = tf.reduce_sum(
        features * pooled_gradients, axis=-1
    )

    cam = tf.nn.relu(cam)

    cam_max = tf.reduce_max(
        cam, axis=1, keepdims=True
    )

    cam = cam / tf.maximum(cam_max, 1e-8)

    cam = tf.image.resize(
        cam[..., tf.newaxis],
        size=(288, 1)
    )

    cam = tf.squeeze(cam, axis=-1)
    cam = tf.squeeze(cam, axis=-1)

    return cam, predictions


def weighted_ce_loss(labels, predictions):
    labels = tf.cast(labels, tf.int32)

    ce = tf.keras.losses.sparse_categorical_crossentropy(
        labels, predictions
    )

    sample_weights = tf.gather(
        class_weights_tensor, labels
    )

    return tf.reduce_mean(ce * sample_weights)


def calculate_b3_loss(
    labels,
    clean_predictions,
    noisy_predictions,
    clean_cam,
    noisy_cam
):
    clean_ce = weighted_ce_loss(
        labels, clean_predictions
    )

    noisy_ce = weighted_ce_loss(
        labels, noisy_predictions
    )

    ce_loss = (clean_ce + noisy_ce) / 2.0

    per_sample_cam_loss = tf.reduce_mean(
        tf.square(clean_cam - noisy_cam),
        axis=1
    )

    cam_loss = tf.reduce_mean(per_sample_cam_loss)

    total_loss = ce_loss + LAMBDA_CAM * cam_loss

    return total_loss, ce_loss, cam_loss


print("CAM function restored!")
print("Weighted CE loss restored!")
print("B3 loss function restored!")

CAM function restored!
Weighted CE loss restored!
B3 loss function restored!


In [51]:
required_variables = [
    "train_b3_step",
    "train_one_epoch",
    "evaluate_noisy_validation",
    "differentiable_cam_for_model",
    "calculate_b3_loss",
    "weighted_ce_loss",
    "b3_feature_extractor",
    "b3_classifier_head",
    "b3_model",
    "b3_optimizer",
    "train_dataset",
    "X_val_noisy",
    "class_weights_tensor"
]

print("=" * 55)
print("FINAL B3 ENVIRONMENT CHECK")
print("=" * 55)

all_available = True

for name in required_variables:
    exists = name in globals()
    print(name, ":", "AVAILABLE" if exists else "MISSING")

    if not exists:
        all_available = False

print("\nTraining batches:", len(train_dataset))
print("Model parameters:", b3_model.count_params())
print("Training history epochs:", len(b3_history))
print("Best validation F1:", best_b3_macro_f1)
print("Learning rate:", float(b3_optimizer.learning_rate.numpy()))

print("\nALL AVAILABLE:", all_available)

FINAL B3 ENVIRONMENT CHECK
train_b3_step : AVAILABLE
train_one_epoch : AVAILABLE
evaluate_noisy_validation : AVAILABLE
differentiable_cam_for_model : AVAILABLE
calculate_b3_loss : AVAILABLE
weighted_ce_loss : AVAILABLE
b3_feature_extractor : AVAILABLE
b3_classifier_head : AVAILABLE
b3_model : AVAILABLE
b3_optimizer : AVAILABLE
train_dataset : AVAILABLE
X_val_noisy : AVAILABLE
class_weights_tensor : AVAILABLE

Training batches: 467
Model parameters: 194020
Training history epochs: 3
Best validation F1: 0.3644898279927664
Learning rate: 9.999999747378752e-05

ALL AVAILABLE: True


In [52]:
# ============================================
# STEP 31: B3 CHECKPOINT & CONFIG VERIFICATION
# ============================================

import os
import pandas as pd
import tensorflow as tf

print("=" * 60)
print("B3 CHECKPOINT VERIFICATION")
print("=" * 60)

checkpoint_paths = {
    "Latest": os.path.join(
        BASE_DIR,
        "models/B3_CAM_Consistency/latest_B3_resnet.keras"
    ),
    "Best": os.path.join(
        BASE_DIR,
        "models/B3_CAM_Consistency/best_B3_resnet.keras"
    ),
    "Initial": os.path.join(
        BASE_DIR,
        "models/B3_CAM_Consistency/initial_B3_from_B2.keras"
    )
}

for name, path in checkpoint_paths.items():
    exists = os.path.exists(path)

    print(f"\n{name}: {'AVAILABLE' if exists else 'MISSING'}")

    if exists:
        print("Path:", path)
        print("Size (MB):", round(os.path.getsize(path) / (1024**2), 2))

print("\n" + "=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

print("Training batches:", len(train_dataset))
print("Dataset element spec:", train_dataset.element_spec)
print("Class weights:", class_weights_tensor.numpy())
print("Learning rate:", float(b3_optimizer.learning_rate.numpy()))
print("CAM lambda:", LAMBDA_CAM)

print("\n" + "=" * 60)
print("TRAINING HISTORY")
print("=" * 60)

print(pd.DataFrame(b3_history))

print("\n" + "=" * 60)
print("VALIDATION CONFIGURATION")
print("=" * 60)

print("Validation samples:", len(X_val))
print("Validation configurations:", len(X_val_noisy))
print("Validation SNRs:", VAL_SNRS)

print("\nCHECKPOINT AND CONFIGURATION CHECK COMPLETED!")

B3 CHECKPOINT VERIFICATION

Latest: AVAILABLE
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/latest_B3_resnet.keras
Size (MB): 0.85

Best: AVAILABLE
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/best_B3_resnet.keras
Size (MB): 0.85

Initial: AVAILABLE
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/initial_B3_from_B2.keras
Size (MB): 0.85

TRAINING CONFIGURATION
Training batches: 467
Dataset element spec: (TensorSpec(shape=(None, 288, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))
Class weights: [ 0.28106704  7.6329246   3.533728   35.54762   ]
Learning rate: 9.999999747378752e-05
CAM lambda: 0.1

TRAINING HISTORY
   epoch  train_loss  train_ce_loss  train_cam_loss  training_time_min  \
0      1    0.402467       0.401468        0.009988          55.045435   
1      2    0.346454       0.345497        0.009563          55.103961   
2      3    0.327933     

In [53]:
# ============================================
# STEP 32: CORRECT BATCH-WISE CAM RESIZING
# ============================================

def differentiable_cam_for_model(
    x, labels, feature_extractor, classifier_head
):
    x = tf.cast(x, tf.float32)
    labels = tf.cast(labels, tf.int32)

    with tf.GradientTape() as tape:
        features = feature_extractor(x, training=False)
        tape.watch(features)

        predictions = classifier_head(features, training=False)

        indices = tf.stack(
            [tf.range(tf.shape(labels)[0]), labels],
            axis=1
        )

        target_scores = tf.gather_nd(predictions, indices)

    gradients = tape.gradient(
        tf.reduce_sum(target_scores), features
    )

    pooled_gradients = tf.reduce_mean(
        gradients, axis=1, keepdims=True
    )

    cam = tf.reduce_sum(
        features * pooled_gradients, axis=-1
    )

    cam = tf.nn.relu(cam)

    cam_max = tf.reduce_max(
        cam, axis=1, keepdims=True
    )

    cam = cam / tf.maximum(cam_max, 1e-8)

    # Correctly preserve batch dimension
    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1)
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam, predictions


print("Corrected batch-wise CAM function ready!")

Corrected batch-wise CAM function ready!


In [54]:
# ============================================
# STEP 33: CAM SHAPE VERIFICATION
# ============================================

test_x, test_y = next(iter(train_dataset))

test_x = test_x[:4]
test_y = test_y[:4]

test_cam, test_predictions = differentiable_cam_for_model(
    test_x,
    test_y,
    b3_feature_extractor,
    b3_classifier_head
)

print("Input shape:", test_x.shape)
print("CAM shape:", test_cam.shape)
print("Prediction shape:", test_predictions.shape)

assert test_cam.shape == (4, 288)
assert test_predictions.shape == (4, 4)

assert bool(
    tf.reduce_all(tf.math.is_finite(test_cam)).numpy()
)

assert bool(
    tf.reduce_all(tf.math.is_finite(test_predictions)).numpy()
)

print("\nCAM SHAPE VERIFICATION PASSED!")

ValueError: Attempt to convert a value (None) with an unsupported type (<class 'NoneType'>) to a Tensor.

In [55]:
# ============================================
# STEP 32: FIX CAM GRADIENT CALCULATION
# ============================================

def differentiable_cam_for_model(
    x, labels, feature_extractor, classifier_head
):
    x = tf.cast(x, tf.float32)
    labels = tf.cast(labels, tf.int32)

    with tf.GradientTape() as tape:

        features = feature_extractor(x, training=False)
        tape.watch(features)

        predictions = classifier_head(
            features, training=False
        )

        indices = tf.stack(
            [
                tf.range(tf.shape(labels)[0]),
                labels
            ],
            axis=1
        )

        target_scores = tf.gather_nd(
            predictions, indices
        )

        # IMPORTANT: Keep this operation inside GradientTape
        target_score_sum = tf.reduce_sum(target_scores)

    gradients = tape.gradient(
        target_score_sum, features
    )

    if gradients is None:
        raise ValueError(
            "CAM gradients are None. Check model connectivity."
        )

    pooled_gradients = tf.reduce_mean(
        gradients, axis=1, keepdims=True
    )

    cam = tf.reduce_sum(
        features * pooled_gradients,
        axis=-1
    )

    cam = tf.nn.relu(cam)

    cam_max = tf.reduce_max(
        cam, axis=1, keepdims=True
    )

    cam = cam / tf.maximum(cam_max, 1e-8)

    # Resize while preserving batch dimension
    cam = cam[:, :, tf.newaxis, tf.newaxis]

    cam = tf.image.resize(
        cam,
        size=(288, 1)
    )

    cam = tf.squeeze(cam, axis=[2, 3])

    return cam, predictions


print("CAM gradient function corrected!")

CAM gradient function corrected!


In [56]:
test_x, test_y = next(iter(train_dataset))

test_x = test_x[:4]
test_y = test_y[:4]

test_cam, test_predictions = differentiable_cam_for_model(
    test_x,
    test_y,
    b3_feature_extractor,
    b3_classifier_head
)

print("Input shape:", test_x.shape)
print("CAM shape:", test_cam.shape)
print("Prediction shape:", test_predictions.shape)
print("CAM gradients finite:",
      bool(tf.reduce_all(tf.math.is_finite(test_cam)).numpy()))

assert test_cam.shape == (4, 288)
assert test_predictions.shape == (4, 4)

print("\nCAM SHAPE VERIFICATION PASSED!")

Input shape: (4, 288, 1)
CAM shape: (4, 288)
Prediction shape: (4, 4)
CAM gradients finite: True

CAM SHAPE VERIFICATION PASSED!


In [57]:
# ============================================
# STEP 34: B3 TRAINING READINESS TEST
# No optimizer update will happen
# ============================================

test_x, test_y = next(iter(train_dataset))

test_x = test_x[:4]
test_y = test_y[:4]

# Generate noisy samples
test_noisy_x, test_snr = tf.map_fn(
    lambda x: add_random_noise_with_snr(x),
    test_x,
    fn_output_signature=(
        tf.TensorSpec(shape=(288, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.float32)
    )
)

# Calculate loss without updating model
with tf.GradientTape() as tape:

    clean_cam, clean_predictions = differentiable_cam_for_model(
        test_x,
        test_y,
        b3_feature_extractor,
        b3_classifier_head
    )

    noisy_cam, noisy_predictions = differentiable_cam_for_model(
        test_noisy_x,
        test_y,
        b3_feature_extractor,
        b3_classifier_head
    )

    total_loss, ce_loss, cam_loss = calculate_b3_loss(
        test_y,
        clean_predictions,
        noisy_predictions,
        clean_cam,
        noisy_cam
    )

gradients = tape.gradient(
    total_loss,
    b3_model.trainable_variables
)

valid_gradients = [
    g for g in gradients if g is not None
]

print("Clean CAM:", clean_cam.shape)
print("Noisy CAM:", noisy_cam.shape)
print("Clean predictions:", clean_predictions.shape)
print("Noisy predictions:", noisy_predictions.shape)

print("\nTotal loss:", float(total_loss.numpy()))
print("CE loss:", float(ce_loss.numpy()))
print("CAM loss:", float(cam_loss.numpy()))

print("\nValid gradients:", len(valid_gradients))
print("Total trainable variables:", len(b3_model.trainable_variables))

assert len(valid_gradients) > 0
assert bool(tf.math.is_finite(total_loss).numpy())
assert bool(tf.math.is_finite(ce_loss).numpy())
assert bool(tf.math.is_finite(cam_loss).numpy())

print("\nB3 TRAINING READINESS TEST PASSED!")

Clean CAM: (4, 288)
Noisy CAM: (4, 288)
Clean predictions: (4, 4)
Noisy predictions: (4, 4)

Total loss: 0.27044662833213806
CE loss: 0.2687036395072937
CAM loss: 0.017429830506443977

Valid gradients: 31
Total trainable variables: 31

B3 TRAINING READINESS TEST PASSED!


In [58]:
# ============================================
# STEP 35: VERIFY ORIGINAL NOTEBOOK CONFIG
# ============================================

import json
import os

notebook_path = os.path.join(
    BASE_DIR,
    "07_CAM_Consistency_Models.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "class_weights_tensor",
    "compute_class_weight",
    "BATCH_SIZE",
    "train_dataset",
    "shuffle(",
    "class_weights"
]

for keyword in keywords:
    print("\n" + "=" * 60)
    print("SEARCH:", keyword)
    print("=" * 60)

    found = False

    for i, cell in enumerate(notebook["cells"]):
        source = "".join(cell.get("source", []))

        if keyword in source:
            found = True
            print(f"\nCELL {i}:")
            print(source[:3500])

    if not found:
        print("Not found.")


SEARCH: class_weights_tensor

CELL 44:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_tensor = tf.constant(
    class_weights_array,
    dtype=tf.float32
)

print("Class weights:", class_weights_array)

print("CLASS WEIGHTS CREATED!")

CELL 45:
LAMBDA_CAM = 0.1

def calculate_b3_loss(
    labels,
    predictions,
    clean_cam,
    noisy_cam
):

    ce_loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        predictions
    )

    sample_weights = tf.gather(
        class_weights_tensor,
        labels
    )

    weighted_ce = tf.reduce_mean(
        ce_loss * sample_weights
    )

    cam_loss = cam_consistency_loss(
        clean_cam,
        noisy_cam
    )

    total_loss = weighted_ce + LAMBDA_CAM * cam_loss

    return total_loss, weighted_ce, cam_loss

CELL 60:
LAMBDA_CAM = 0.1

def we

In [59]:
# ============================================
# STEP 36: VERIFY BATCH SIZE AND HISTORY
# ============================================

import math
import os
import pandas as pd

print("CURRENT DATASET")
print("Samples:", len(X_train))
print("Batches:", len(train_dataset))

current_batch_size = train_dataset.element_spec[0].shape[0]

print("Dataset element batch dimension:", current_batch_size)

print("\nPOSSIBLE BATCH CONFIGURATIONS")

for bs in [8, 32, 128]:
    print(
        f"Batch size {bs}:",
        math.ceil(len(X_train) / bs),
        "batches"
    )

print("\nSAVED HISTORY")

history_path = os.path.join(
    BASE_DIR,
    "reports/B3_training_history.csv"
)

if os.path.exists(history_path):
    saved_history = pd.read_csv(history_path)
    print(saved_history)
else:
    print("History CSV not found.")

print("\nIN-MEMORY HISTORY")
print(pd.DataFrame(b3_history))

print("\nCHECKPOINT TIMESTAMPS")

for name, path in checkpoint_paths.items():
    if os.path.exists(path):
        print(
            name,
            "modified:",
            pd.Timestamp(os.path.getmtime(path), unit="s")
        )

CURRENT DATASET
Samples: 59720
Batches: 467
Dataset element batch dimension: None

POSSIBLE BATCH CONFIGURATIONS
Batch size 8: 7465 batches
Batch size 32: 1867 batches
Batch size 128: 467 batches

SAVED HISTORY
   epoch  train_loss  train_ce_loss  train_cam_loss  training_time_min  \
0      1    0.402467       0.401468        0.009988          55.045435   
1      2    0.346454       0.345497        0.009563          55.103961   
2      3    0.327933       0.327065        0.008683          54.883796   

   val_macro_f1  
0      0.303642  
1      0.343436  
2      0.364490  

IN-MEMORY HISTORY
   epoch  train_loss  train_ce_loss  train_cam_loss  training_time_min  \
0      1    0.402467       0.401468        0.009988          55.045435   
1      2    0.346454       0.345497        0.009563          55.103961   
2      3    0.327933       0.327065        0.008683          54.883796   

   val_macro_f1  
0      0.303642  
1      0.343436  
2      0.364490  

CHECKPOINT TIMESTAMPS
Latest mo

In [60]:
# ============================================
# STEP 37: B3 EPOCH 4 TRAINING
# ============================================

import os
import pandas as pd
import time

epoch = 4

# Verify previous training history
assert len(b3_history) == 3
assert b3_history[-1]["epoch"] == 3

# Verify checkpoints
assert os.path.exists(B3_LATEST_PATH)
assert os.path.exists(B3_CHECKPOINT_PATH)

# Verify dataset
assert len(train_dataset) == 467

print("=" * 60)
print("STARTING B3 EPOCH 4")
print("=" * 60)

print("Training samples:", len(X_train))
print("Training batches:", len(train_dataset))
print("Model parameters:", b3_model.count_params())
print("Learning rate:", float(b3_optimizer.learning_rate.numpy()))
print("Previous best Macro-F1:", best_b3_macro_f1)

# --------------------------------------------
# TRAINING
# --------------------------------------------

start_time = time.time()

train_result = train_one_epoch(
    train_b3_step,
    train_dataset,
    b3_model,
    "B3"
)

print("\nEPOCH 4 TRAINING FINISHED!")

print("Training loss:", train_result["train_loss"])
print("CE loss:", train_result["train_ce_loss"])
print("CAM loss:", train_result["train_cam_loss"])
print("Training time (minutes):", train_result["training_time_min"])

# --------------------------------------------
# VALIDATION
# --------------------------------------------

print("\nEvaluating B3 on noisy validation data...")

val_macro_f1 = evaluate_noisy_validation(b3_model)

# --------------------------------------------
# HISTORY
# --------------------------------------------

epoch_result = {
    "epoch": epoch,
    **train_result,
    "val_macro_f1": val_macro_f1
}

b3_history.append(epoch_result)

# --------------------------------------------
# SAVE LATEST CHECKPOINT
# --------------------------------------------

latest_temp = B3_LATEST_PATH.replace(
    ".keras", "_temp.keras"
)

b3_model.save(latest_temp)
os.replace(latest_temp, B3_LATEST_PATH)

print("\nLatest checkpoint saved!")

# --------------------------------------------
# SAVE BEST CHECKPOINT IF IMPROVED
# --------------------------------------------

if val_macro_f1 > best_b3_macro_f1:

    best_temp = B3_CHECKPOINT_PATH.replace(
        ".keras", "_temp.keras"
    )

    b3_model.save(best_temp)
    os.replace(best_temp, B3_CHECKPOINT_PATH)

    best_b3_macro_f1 = val_macro_f1

    print("NEW BEST B3 MODEL SAVED!")

else:
    print("Previous best checkpoint preserved.")

# --------------------------------------------
# SAVE HISTORY
# --------------------------------------------

history_temp = B3_HISTORY_PATH.replace(
    ".csv", "_temp.csv"
)

pd.DataFrame(b3_history).to_csv(
    history_temp,
    index=False
)

os.replace(history_temp, B3_HISTORY_PATH)

# --------------------------------------------
# FINAL REPORT
# --------------------------------------------

print("\n" + "=" * 60)
print("B3 EPOCH 4 COMPLETED!")
print("=" * 60)

print("Epoch:", epoch)
print("Validation Macro-F1:", val_macro_f1)
print("Best Macro-F1:", best_b3_macro_f1)

print("\nTraining history:")
print(pd.DataFrame(b3_history))

print("\nLatest checkpoint exists:",
      os.path.exists(B3_LATEST_PATH))

print("Best checkpoint exists:",
      os.path.exists(B3_CHECKPOINT_PATH))

print("History CSV exists:",
      os.path.exists(B3_HISTORY_PATH))

print("\nTOTAL TIME (minutes):",
      round((time.time() - start_time) / 60, 2))

print("\nB3 EPOCH 4 FINISHED SUCCESSFULLY!")

STARTING B3 EPOCH 4
Training samples: 59720
Training batches: 467
Model parameters: 194020
Learning rate: 9.999999747378752e-05
Previous best Macro-F1: 0.3644898279927664

EPOCH 4 TRAINING FINISHED!
Training loss: 0.29901446319909114
CE loss: 0.2981343682121477
CAM loss: 0.008800946865569825
Training time (minutes): 9.787625741958617

Evaluating B3 on noisy validation data...
BW | 18 dB | Macro-F1: 0.3427
BW | 12 dB | Macro-F1: 0.3434
BW | 6 dB | Macro-F1: 0.3598
MA | 18 dB | Macro-F1: 0.3453
MA | 12 dB | Macro-F1: 0.3475
MA | 6 dB | Macro-F1: 0.3494

Latest checkpoint saved!
Previous best checkpoint preserved.

B3 EPOCH 4 COMPLETED!
Epoch: 4
Validation Macro-F1: 0.34801116820149897
Best Macro-F1: 0.3644898279927664

Training history:
   epoch  train_loss  train_ce_loss  train_cam_loss  training_time_min  \
0      1    0.402467       0.401468        0.009988          55.045435   
1      2    0.346454       0.345497        0.009563          55.103961   
2      3    0.327933       0.3270

In [61]:
# ============================================
# STEP 38: VERIFY EPOCH 4 ENVIRONMENT
# ============================================

import os
import tensorflow as tf
import pandas as pd

print("=" * 55)
print("GPU STATUS")
print("=" * 55)

print(tf.config.list_physical_devices("GPU"))

print("\n" + "=" * 55)
print("ACTUAL TRAINING BATCH")
print("=" * 55)

sample_x, sample_y = next(iter(train_dataset))

print("Input batch shape:", sample_x.shape)
print("Label batch shape:", sample_y.shape)

print("\n" + "=" * 55)
print("TRAINING HISTORY")
print("=" * 55)

print(pd.read_csv(B3_HISTORY_PATH))

print("\n" + "=" * 55)
print("CHECKPOINT INFORMATION")
print("=" * 55)

for name, path in [
    ("LATEST", B3_LATEST_PATH),
    ("BEST", B3_CHECKPOINT_PATH)
]:

    print("\n", name)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print(
            "Size (MB):",
            round(os.path.getsize(path) / (1024 * 1024), 3)
        )

        print(
            "Modified:",
            pd.Timestamp(os.path.getmtime(path), unit="s")
        )

print("\n" + "=" * 55)
print("BEST MODEL MACRO-F1:", best_b3_macro_f1)
print("LATEST COMPLETED EPOCH:", b3_history[-1]["epoch"])
print("=" * 55)

GPU STATUS
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

ACTUAL TRAINING BATCH
Input batch shape: (128, 288, 1)
Label batch shape: (128,)

TRAINING HISTORY
   epoch  train_loss  train_ce_loss  train_cam_loss  training_time_min  \
0      1    0.402467       0.401468        0.009988          55.045435   
1      2    0.346454       0.345497        0.009563          55.103961   
2      3    0.327933       0.327065        0.008683          54.883796   
3      4    0.299014       0.298134        0.008801           9.787626   

   val_macro_f1  
0      0.303642  
1      0.343436  
2      0.364490  
3      0.348011  

CHECKPOINT INFORMATION

 LATEST
Exists: True
Size (MB): 0.854
Modified: 2026-11-21 16:28:09

 BEST
Exists: True
Size (MB): 0.854
Modified: 2026-11-20 15:53:33

BEST MODEL MACRO-F1: 0.3644898279927664
LATEST COMPLETED EPOCH: 4


In [62]:
# ============================================
# STEP 39: GPU PERFORMANCE DIAGNOSTIC
# ============================================

import tensorflow as tf

print("=" * 55)
print("GPU DETAILS")
print("=" * 55)

!nvidia-smi

print("\n" + "=" * 55)
print("TENSORFLOW GPU DETAILS")
print("=" * 55)

for gpu in tf.config.list_physical_devices("GPU"):
    print(tf.config.experimental.get_device_details(gpu))

print("\n" + "=" * 55)
print("DATASET VERIFICATION")
print("=" * 55)

print("Dataset batches:", len(train_dataset))
print("Dataset cardinality:", tf.data.experimental.cardinality(
    train_dataset
).numpy())

print("Actual batch shape:", next(iter(train_dataset))[0].shape)

print("\n" + "=" * 55)
print("MODEL STATUS")
print("=" * 55)

print("Parameters:", b3_model.count_params())
print("Latest epoch:", b3_history[-1]["epoch"])
print("Best Macro-F1:", best_b3_macro_f1)

print("\nDIAGNOSTIC COMPLETED!")

GPU DETAILS
Sat Nov 21 16:38:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             27W /   70W |   12683MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

In [63]:
# ============================================
# STEP 40: BACKUP B3 CHECKPOINTS
# ============================================

import os
import shutil
from datetime import datetime

# Backup directory
BACKUP_DIR = os.path.join(
    BASE_DIR,
    "models",
    "B3_CAM_Consistency",
    "backups"
)

os.makedirs(BACKUP_DIR, exist_ok=True)

# Unique timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_files = {
    "latest": B3_LATEST_PATH,
    "best": B3_CHECKPOINT_PATH,
    "history": B3_HISTORY_PATH
}

print("=" * 55)
print("B3 BACKUP PROCESS")
print("=" * 55)

for name, source in backup_files.items():

    if not os.path.exists(source):
        raise FileNotFoundError(
            f"Missing source file: {source}"
        )

    extension = os.path.splitext(source)[1]

    destination = os.path.join(
        BACKUP_DIR,
        f"{name}_epoch4_{timestamp}{extension}"
    )

    shutil.copy2(source, destination)

    print(f"\n{name.upper()} BACKUP CREATED")
    print("Path:", destination)
    print(
        "Size (MB):",
        round(os.path.getsize(destination) / (1024 * 1024), 3)
    )

print("\n" + "=" * 55)
print("BACKUP VERIFICATION")
print("=" * 55)

for name in backup_files:

    matches = [
        f for f in os.listdir(BACKUP_DIR)
        if f.startswith(f"{name}_epoch4_{timestamp}")
    ]

    print(name.upper(), ":", "VERIFIED" if matches else "FAILED")

print("\nB3 EPOCH 4 BACKUP COMPLETED!")

B3 BACKUP PROCESS

LATEST BACKUP CREATED
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/backups/latest_epoch4_20261121_163837.keras
Size (MB): 0.854

BEST BACKUP CREATED
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/backups/best_epoch4_20261121_163837.keras
Size (MB): 0.854

HISTORY BACKUP CREATED
Path: /content/gdrive_mount/MyDrive/ECG_QPAIN_2027/models/B3_CAM_Consistency/backups/history_epoch4_20261121_163837.csv
Size (MB): 0.0

BACKUP VERIFICATION
LATEST : VERIFIED
BEST : VERIFIED
HISTORY : VERIFIED

B3 EPOCH 4 BACKUP COMPLETED!


In [64]:
# ============================================
# STEP 41: B3 EPOCH 5 TRAINING
# ============================================

import os
import time
import pandas as pd

epoch = 5

# Safety checks
assert b3_history[-1]["epoch"] == 4
assert len(b3_history) == 4
assert os.path.exists(B3_LATEST_PATH)
assert os.path.exists(B3_CHECKPOINT_PATH)
assert len(train_dataset) == 467

# Verify current batch
sample_x, sample_y = next(iter(train_dataset))

assert sample_x.shape == (128, 288, 1)
assert sample_y.shape == (128,)

print("=" * 55)
print("STARTING B3 EPOCH 5")
print("=" * 55)

print("GPU:", tf.config.list_physical_devices("GPU"))
print("Training samples:", len(X_train))
print("Training batches:", len(train_dataset))
print("Batch shape:", sample_x.shape)
print("Previous best Macro-F1:", best_b3_macro_f1)

# --------------------------------------------
# TRAINING
# --------------------------------------------

start_time = time.time()

train_result = train_one_epoch(
    train_b3_step,
    train_dataset,
    b3_model,
    "B3"
)

training_elapsed = time.time() - start_time

print("\nEPOCH 5 TRAINING COMPLETED")

print("Training loss:", train_result["train_loss"])
print("CE loss:", train_result["train_ce_loss"])
print("CAM loss:", train_result["train_cam_loss"])
print("Training time:", round(training_elapsed / 60, 2), "minutes")

# --------------------------------------------
# VALIDATION
# --------------------------------------------

print("\nEVALUATING NOISY VALIDATION DATA")

val_macro_f1 = evaluate_noisy_validation(b3_model)

# --------------------------------------------
# UPDATE HISTORY
# --------------------------------------------

epoch_result = {
    "epoch": epoch,
    **train_result,
    "val_macro_f1": val_macro_f1
}

b3_history.append(epoch_result)

# --------------------------------------------
# SAVE LATEST
# --------------------------------------------

latest_temp = B3_LATEST_PATH.replace(
    ".keras", "_temp.keras"
)

b3_model.save(latest_temp)
os.replace(latest_temp, B3_LATEST_PATH)

print("\nLATEST CHECKPOINT SAVED")

# --------------------------------------------
# UPDATE BEST
# --------------------------------------------

if val_macro_f1 > best_b3_macro_f1:

    best_temp = B3_CHECKPOINT_PATH.replace(
        ".keras", "_temp.keras"
    )

    b3_model.save(best_temp)
    os.replace(best_temp, B3_CHECKPOINT_PATH)

    best_b3_macro_f1 = val_macro_f1

    print("NEW BEST MODEL SAVED!")

else:
    print("BEST MODEL PRESERVED")

# --------------------------------------------
# SAVE HISTORY
# --------------------------------------------

history_temp = B3_HISTORY_PATH.replace(
    ".csv", "_temp.csv"
)

pd.DataFrame(b3_history).to_csv(
    history_temp,
    index=False
)

os.replace(history_temp, B3_HISTORY_PATH)

# --------------------------------------------
# REPORT
# --------------------------------------------

print("\n" + "=" * 55)
print("B3 EPOCH 5 COMPLETED")
print("=" * 55)

print("Validation Macro-F1:", val_macro_f1)
print("Best Macro-F1:", best_b3_macro_f1)

print("\nUPDATED TRAINING HISTORY")
print(pd.DataFrame(b3_history))

print("\nB3 EPOCH 5 FINISHED SUCCESSFULLY!")

STARTING B3 EPOCH 5
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training samples: 59720
Training batches: 467
Batch shape: (128, 288, 1)
Previous best Macro-F1: 0.3644898279927664

EPOCH 5 TRAINING COMPLETED
Training loss: 0.29190398723174216
CE loss: 0.2910846733992692
CAM loss: 0.008193133974236454
Training time: 9.76 minutes

EVALUATING NOISY VALIDATION DATA
BW | 18 dB | Macro-F1: 0.3496
BW | 12 dB | Macro-F1: 0.3487
BW | 6 dB | Macro-F1: 0.3653
MA | 18 dB | Macro-F1: 0.3533
MA | 12 dB | Macro-F1: 0.3551
MA | 6 dB | Macro-F1: 0.3535

LATEST CHECKPOINT SAVED
BEST MODEL PRESERVED

B3 EPOCH 5 COMPLETED
Validation Macro-F1: 0.3542619613487781
Best Macro-F1: 0.3644898279927664

UPDATED TRAINING HISTORY
   epoch  train_loss  train_ce_loss  train_cam_loss  training_time_min  \
0      1    0.402467       0.401468        0.009988          55.045435   
1      2    0.346454       0.345497        0.009563          55.103961   
2      3    0.327933       0.327065     